# Prediksi Beban Listrik per Jam Satu Hari ke Depan (*Day-Ahead Load Forecasting*) dan Segmentasi Profil Beban Harian pada Jaringan PJM
**Laporan Mini-Project *Machine Learning* · Kecerdasan Artifisial (TKEE262513)** &nbsp;|&nbsp; **Kelompok 14:** Alfito Putra Parindra (24/534768/TK/59290) · Muhammad Fasha (24/535041/TK/59325) · Najamudin Fawwaz Haq (24/539273/TK/59804) · Qais Faqih Basyaev (24/542181/TK/60189) · Muhammad Zeva Hardi Seswanto (24/542780/TK/60292) &nbsp;|&nbsp; Departemen Teknik Elektro dan Teknologi Informasi, Fakultas Teknik, Universitas Gadjah Mada, 2026

Dua pendekatan AI pada satu data beban per jam jaringan PJM (Amerika Serikat, 2002–2018): ***regression*** (HistGradientBoosting, dibandingkan dengan Random Forest, Ridge, dan *baseline* sederhana) untuk memprediksi beban tiap jam **sehari ke depan**, dan ***clustering*** (K-Means, dipilih lewat *tuning preprocessing* dan perbandingan algoritma) untuk mengelompokkan profil beban harian. PJM East (PJME) menjadi analisis utama, lalu pipeline yang sama dijalankan pada 9 wilayah PJM lain.

Notebook ini mengikuti struktur laporan: **Bab 1** Pendahuluan, **Bab 2** Metode dan Model AI (*AI formulation* dan *data/knowledge preparation*), **Bab 3** Hasil dan Pembahasan (*implementation*, *results*, *evaluation*, *engineering interpretation*), dan **Bab 4** Kesimpulan. Pada *implementation* dan *results*, *regression* dibahas lebih dulu karena merupakan model utama, dan *clustering* menyusul sebagai analisis pendukung.

**Cara menjalankan** (dari *root* repo; Python 3.10+ dan paket di `requirements.txt`): buka notebook ini lalu *Run All* (sekitar 40 menit pada 2 CPU). Data dibaca dari `data_pjm/`, hasil *tuning clustering* penuh dari `cluster_tuning/` (dihasilkan terpisah oleh skrip di folder itu), dan semua keluaran (JSON, gambar) ditulis ke `results/`.

> **Catatan.** Beban agregat wilayah jauh lebih teratur daripada konsumsi satu rumah tangga. Akurasi yang tinggi di sini terutama berasal dari sifat data (agregat, per jam, riwayat sangat berkorelasi), bukan dari model yang canggih (lihat Subbab 3.4).


## BAB 1. PENDAHULUAN
### 1.1 Latar Belakang
Listrik merupakan infrastruktur yang menopang hampir seluruh kegiatan ekonomi dan sosial, dan kebutuhannya di Indonesia terus bertambah. Konsumsi listrik per kapita Indonesia mencapai 1.285 kWh pada Oktober 2023 [1], sedangkan ESDM menyebut rata-rata ASEAN sebesar 3.672 kWh per kapita [2]. Pemerintah menargetkan tambahan kapasitas pembangkit 69,5 GW dalam RUPTL 2025–2034 [3], dan PLN memproyeksikan kebutuhan listrik naik dari 306 TWh menjadi 511 TWh pada 2034 [4]. Pertumbuhan ini harus dikelola oleh operator sistem yang setiap saat menyeimbangkan daya yang dibangkitkan dengan daya yang dikonsumsi.

Sistem Jawa-Bali menunjukkan skala masalahnya. Beban puncak malam sistem ini tercatat 27.070 MW pada 17 Oktober 2018 [5], dan kelebihan pasok (*oversupply*) turun dari 7 GW pada akhir 2022 menjadi 4 GW pada akhir 2023 [6]. Prediksi beban yang terlalu rendah berisiko membuat cadangan daya tidak mencukupi, sedangkan prediksi yang terlalu tinggi membuat pembangkit beroperasi lebih banyak dari kebutuhan. Di sisi lain, 42,6 GW dari 69,5 GW kapasitas baru dalam RUPTL berasal dari energi terbarukan dan 10,3 GW dari penyimpanan energi [3], sehingga perencanaan operasi harian makin menuntut prediksi beban yang akurat. Nilai ekonominya pun nyata: perkiraan kasar menyebut penurunan *error* prediksi beban jangka pendek sebesar 1 poin persentase (MAPE dari 4% menjadi 3%) menghemat sekitar US$300.000 per tahun untuk utilitas dengan beban puncak 1 GW [7].

Prediksi beban jangka pendek dengan *machine learning* sudah diteliti pada sistem Indonesia. Ainuri dkk. memakai *Long Short-Term Memory* (LSTM) pada sistem Jawa-Bali dan memperoleh MAPE 5,67% serta RMSE 1.683,09 MW pada model terbaiknya [8]. Namun, dalam penelusuran kami tidak ditemukan data beban per jam sistem Indonesia yang dapat diunduh secara publik, sehingga metode sulit direproduksi dan dibandingkan secara terbuka. Sebagai gantinya, data beban per jam PJM Interconnection tersedia secara publik [9]–[13]. PJM melayani 65 juta penduduk di 13 negara bagian dan District of Columbia, dengan perkiraan beban puncak musim panas sekitar 149.000 MW dan kapasitas terpasang sekitar 185.000 MW [14]. Skala ini cukup besar untuk menguji metode pada beberapa wilayah dengan karakter beban berbeda.

Berdasarkan uraian tersebut, mini project ini membangun model prediksi beban *day-ahead* per jam dengan HistGradientBoosting (HGB) pada data PJM, dan mengelompokkan hari berdasarkan profil beban 24 jam dengan K-Means. Model diuji pada sepuluh wilayah dengan pembagian data kronologis dan dibandingkan dengan *baseline* sederhana, sehingga peningkatan akurasinya dapat diukur secara terbuka. Hasilnya diharapkan menjadi acuan metode yang dapat diterapkan pada data beban sistem Indonesia ketika data tersebut tersedia.

### 1.2 Input, Output, dan Perangkat Uji Coba
Masukan, keluaran, dan perangkat uji coba sistem pada setiap pendekatan AI disajikan dalam Tabel 1 sebagai berikut.

*Tabel 1. Input, output, dan perangkat uji coba sistem.*

| Komponen | Keterangan |
|---|---|
| Input (*regression*) | Informasi kalender, meliputi jam, hari dalam pekan, bulan, dan hari libur nasional Amerika Serikat, serta riwayat beban yang tersedia hingga pukul 23:00 hari D−1. |
| Output (*regression*) | Sebanyak 24 nilai prediksi beban dalam MW, masing-masing untuk setiap jam pada hari D. |
| Input (*clustering*) | Satu profil beban yang terdiri atas 24 nilai per jam untuk setiap hari. |
| Output (*clustering*) | Label *cluster* setiap hari dan uraian karakteristik masing-masing *cluster*. |
| Bahasa pemrograman | Python 3.11 |
| *Library* | pandas 3.0, NumPy 2.4, Matplotlib 3.10, scikit-learn 1.8 [15], *holidays* 0.106 [16] |
| *Platform* | Jupyter Notebook (`pjm_load_forecasting.ipynb`) |
| Sistem operasi | Linux |
| Perangkat keras | 2 CPU, RAM 7 GB, tanpa GPU |
| *Random seed* | 42 |


### 1.3 Tujuan Penggunaan AI
Beban listrik dipengaruhi oleh pola harian, mingguan, dan tahunan, serta keberadaan hari libur. Pengaruh berbagai pola tersebut terhadap besarnya beban tidak bersifat linear dan dapat saling berinteraksi. Sebagai contoh, profil beban harian pada musim panas dapat berbeda dari musim dingin, sebagaimana profil hari kerja dapat berbeda dari akhir pekan. Keragaman kombinasi tersebut sulit diwakili secara menyeluruh melalui aturan IF–THEN.

*Machine learning* memungkinkan hubungan tersebut dipelajari dari data historis dalam jumlah besar. Pada wilayah PJME, tersedia 145.366 baris data mentah yang digunakan sebagai dasar analisis. Dengan pendekatan ini, pola beban diidentifikasi melalui pembelajaran dari data, sehingga tidak bergantung pada penyusunan aturan secara manual.

### 1.4 Batasan Masalah
Batasan ruang lingkup mini project ini ditetapkan sebagai berikut.
1. Informasi yang digunakan model terbatas pada kalender dan riwayat beban. Karena *dataset* tidak menyediakan data cuaca, pengaruh suhu tidak dimodelkan.
2. Prediksi dibuat satu kali setiap hari, yaitu pukul 00:00, untuk mencakup 24 jam pada hari tersebut (*day-ahead*).
3. Analisis dilakukan pada 10 wilayah PJM. Seri NI dan PJM_Load yang juga tersedia dalam paket data tidak disertakan karena masing-masing berakhir pada Januari 2011 dan Januari 2002, sehingga tidak mencakup periode *testing*.
4. Data yang dianalisis berasal dari Amerika Serikat. Oleh karena itu, hasil yang diperoleh belum tentu dapat diterapkan secara langsung pada sistem kelistrikan Indonesia.
5. Keluaran model berupa satu nilai prediksi untuk setiap jam, tanpa menyertakan *prediction interval*.


## BAB 2. METODE DAN MODEL AI
### 2.1 AI Formulation
Pendekatan kecerdasan artifisial yang digunakan pada proyek adalah *regression* dalam prediksi beban dengan keluaran yang diprediksi berupa nilai numerik kontinu, yaitu beban listrik dalam MW. Sementara itu, digunakan *clustering* (K-Means) sebagai analisis pendukung untuk mengelompokkan profil beban harian tanpa label kelas. Hasil *clustering* dipakai untuk membantu menjelaskan karakteristik data dan tidak dipakai sebagai masukan model *regression*. Formulasi AI yang digunakan pada proyek ditunjukkan pada Tabel 2 sebagai berikut.

*Tabel 2. Formulasi AI yang digunakan.*

| Tujuan | Pendekatan | Masukan/Input | Keluaran/Output dan Evaluasi |
|---|---|---|---|
| Prediksi beban *day-ahead* | *Regression*: HistGradientBoosting (HGB) dengan pembanding Random Forest, Ridge, dan empat *baseline*. | Kalender dan riwayat beban sampai 23:00 hari D-1 (21 *feature*) | Beban (MW) tiap jam pada hari D; MAE, RMSE, R², *relative error* |
| Segmentasi profil beban harian | *Clustering*: K-Means pada profil dalam MW tanpa standardisasi, hasil *tuning preprocessing* dan perbandingan 6 algoritma; jumlah *cluster* (k) dipilih dengan Silhouette Score | Profil 24 jam per hari dalam MW | Label *cluster*; Silhouette Score, Davies-Bouldin Index, stabilitas, R² rekonstruksi |

Pendekatan utama yang dipilih adalah *regression*, dengan HGB sebagai model utama karena data berbentuk tabel dengan banyak *feature* campuran dan jumlah data yang cukup besar, serta mampu menangkap interaksi antar-*feature* tanpa rekayasa manual. Input model berupa kalender dan riwayat beban sampai pukul 23:00 hari D-1, sedangkan output berupa beban listrik tiap jam pada hari D. HGB dibandingkan dengan Random Forest dan Ridge Regression untuk melihat apakah peningkatan akurasi berasal dari algoritma atau dari *feature* yang digunakan. Empat *baseline* tanpa proses *learning* juga dipakai sebagai pembanding minimum, sehingga ketika model tidak lebih baik dari *baseline* tersebut, penggunaan *machine learning* tidak memberikan manfaat. Pada proyek ini, satu model dipakai untuk seluruh jam dalam sehari, dengan jam sebagai salah satu *feature*.

Pendekatan pendukung adalah *clustering* dengan *unsupervised learning*, karena data beban tidak memiliki label jenis hari sehingga mampu untuk meringkas pola beban harian dalam pengelompokan sesuai dengan kemiripan bentuk profil. Input berupa profil beban 24 jam per hari dan output berupa label *cluster* untuk tiap hari dengan algoritma yang dipakai adalah K-Means. K-Means mengelompokkan setiap hari ke pusat *cluster* (*centroid*) terdekat dengan *centroid* berupa rata-rata profil beban dalam MW sehingga setiap *cluster* mudah dibaca dan dikaitkan dengan musim atau akhir pekan. Pada proyek ini, jumlah *cluster* (k) ditentukan dari Silhouette Score.

### 2.2 Data/Knowledge Preparation
#### 2.2.1 Sumber dan Karakteristik Dataset


Data yang digunakan pada proyek ini adalah data riil/nyata dari *open-public sources* sebagai berikut.
1. Repositori GitHub: panambY (Hourly_Energy_Consumption) [13]
2. Repositori Kaggle: robikscube (Hourly Energy Consumption) [9]
3. Repositori Kaggle: ashishkumarrana (PJM Hourly Energy Consumption) [10]
4. Repositori GitHub: navyasancheti (PJME-Hourly-Energy-Consumption) [12]
5. IEEE DataPort: PJM Hourly Energy Consumption [11]

Berkas *PJME_hourly.csv* dibaca dari repositori panambY [13], sedangkan berkas sembilan wilayah lainnya berasal dari paket data yang diunggah tim. README pada [13] menyatakan bahwa data berasal dari situs PJM, dinyatakan dalam satuan MW, dan cakupan wilayahnya berubah dari tahun ke tahun. Data **tidak** diambil langsung dari PJM dan tidak diverifikasi ke sumber aslinya. Zona waktu pada *timestamp* tidak dijelaskan di sumber sehingga diperlakukan sebagai waktu lokal. Nama lengkap tiap kode wilayah juga tidak dijelaskan di sumber, sehingga ditulis sebagai kode saja.

Setiap berkas terdiri atas dua kolom, yaitu *Datetime* (*timestamp* per jam) dan beban listrik dalam MW. Pada PJME terdapat 145.366 baris mentah, yang mencakup 4 *timestamp* ganda dan 30 jam yang hilang (*missing value*), seluruhnya berupa celah sepanjang 1 jam. Data PJME yang dipakai mencakup 2002-01-02 sampai 2018-08-02 dengan 145.368 jam (6.057 hari). Beban rata-rata 32.079 MW dengan simpangan baku 6.465 MW, nilai minimum 14.544 MW, dan nilai maksimum 62.009 MW. Sel di bawah memuat dan membersihkan data PJME; keluarannya menampilkan angka-angka tersebut.


In [1]:
import json, os, warnings, time
os.makedirs("results", exist_ok=True)   # semua keluaran (JSON, gambar) disimpan di folder results/
import numpy as np, pandas as pd, holidays
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "font.size": 10})
RES = {}          # semua angka hasil dikumpulkan di sini -> results.json (dipakai laporan)
SEED = 42


In [2]:
raw = pd.read_csv("data_pjm/PJME_hourly.csv", parse_dates=["Datetime"]).set_index("Datetime").sort_index()
print("baris mentah:", len(raw), "|", raw.index.min(), "->", raw.index.max())
print("timestamp ganda:", raw.index.duplicated().sum())
y = raw["PJME_MW"]
y = y[~y.index.duplicated(keep="first")]                      # tanggal ganda (pergantian DST) -> ambil yang pertama
full = pd.date_range(y.index.min(), y.index.max(), freq="h")
missing = full.difference(y.index)
print("jam hilang:", len(missing), "| gap terpanjang (jam):",
      int(pd.Series(missing).diff().ne(pd.Timedelta("1h")).cumsum().value_counts().max()) if len(missing) else 0)
y = y.reindex(full).interpolate(limit=6)                      # interpolasi linear hanya untuk gap kecil
y = y["2002-01-02":"2018-08-02"]                              # hanya hari penuh (00:00-23:00)
print("sisa NaN:", int(y.isna().sum()), "| jam akhir:", len(y), "| hari:", len(y)//24)
print(y.describe().round(0))
RES["data"] = dict(rows_raw=int(len(raw)), dup=int(raw.index.duplicated().sum()), missing=int(len(missing)),
                   hours=int(len(y)), days=int(len(y)//24), start=str(y.index[0].date()), end=str(y.index[-1].date()),
                   mean=float(y.mean()), std=float(y.std()), mn=float(y.min()), mx=float(y.max()))


baris mentah: 145366 | 2002-01-01 01:00:00 -> 2018-08-03 00:00:00
timestamp ganda: 4
jam hilang: 30 | gap terpanjang (jam): 1
sisa NaN: 0 | jam akhir: 145368 | hari: 6057
count    145368.0
mean      32079.0
std        6465.0
min       14544.0
25%       27571.0
50%       31420.0
75%       35649.0
max       62009.0
Name: PJME_MW, dtype: float64


#### 2.2.2 Pra-pemrosesan Data
Pra-pemrosesan data atau *preprocessing* dilakukan dengan aturan yang sama pada semua wilayah. Pertama, *duplicate timestamp* yang muncul akibat pergantian *daylight saving time* diambil yang pertama. Kedua, data diubah menjadi *hourly time series* tanpa *gap*, sehingga jam yang hilang menjadi *missing value*. Ketiga, *outlier*, yaitu nilai nol, nilai negatif, atau nilai di bawah 35% dari *local median* pada *hour-of-week* yang sama (*window* 9 minggu), diubah menjadi *missing value*. Keempat, *gap* sampai 6 jam diisi dengan *linear interpolation*. Kelima, hanya *complete day* (24 jam penuh) yang dipakai. Nilai tinggi tidak dibuang karena *peak load* yang ekstrem merupakan kondisi nyata yang justru perlu diprediksi. Nilai di antara 35% dan 50% dari *local median* dicatat sebagai *suspicious value*, tetapi tetap dipertahankan karena tidak dapat dibuktikan salah. Hasilnya untuk setiap wilayah dihitung oleh sel berikut dan diringkas pada Tabel 3.


In [3]:
import os
DATA_DIR = "data_pjm"        # folder berisi <WILAYAH>_hourly.csv (kolom Datetime + beban MW)
REGIONS = ["PJME", "AEP", "COMED", "DAYTON", "DEOK", "DOM", "DUQ", "EKPC", "FE", "PJMW"]
EXCLUDED = {"NI": "seri berhenti Januari 2011 (cakupan wilayah berbeda dari seri lain)", "PJM_Load": "seri berhenti Januari 2002 (cakupan wilayah berbeda)"}
SPLIT_MR = "2016-01-01"      # periode uji bersama semua wilayah: 2016-01-01 s.d. 2018-08-02
LOW_FRAC, BORDER_FRAC = 0.35, 0.50
RES2 = {"split": SPLIT_MR, "excluded": EXCLUDED, "regions": {}}
os.makedirs("results/regions_out", exist_ok=True)


In [4]:
def load_raw(code):
    s = pd.read_csv(f"{DATA_DIR}/{code}_hourly.csv", parse_dates=["Datetime"]).set_index("Datetime").sort_index()
    return s.iloc[:, 0].astype(float)

def clean_region(code):
    """Pra-proses satu wilayah: stempel ganda -> ambil pertama; jam hilang -> NaN; nilai tak masuk akal (<=0 atau < 35% median
    jam-dalam-pekan lokal) -> NaN; gap <= 6 jam diinterpolasi linear; hanya hari penuh dipakai. Mengembalikan (seri, log)."""
    s = load_raw(code); n_raw = int(len(s)); n_dup = int(s.index.duplicated().sum()); s = s[~s.index.duplicated(keep="first")]
    full = pd.date_range(s.index.min(), s.index.max(), freq="h"); yy = s.reindex(full); n_missing = int(yy.isna().sum())
    how = yy.index.dayofweek * 24 + yy.index.hour
    med = yy.groupby(how).transform(lambda x: x.rolling(9, center=True, min_periods=3).median()); ratio = yy / med
    low = ((yy <= 0) | (ratio < LOW_FRAC)).fillna(False); border = ((ratio >= LOW_FRAC) & (ratio < BORDER_FRAC)).fillna(False)
    anomalies = [dict(t=str(i), nilai=float(yy[i]), rasio=float(ratio[i]) if np.isfinite(ratio[i]) else 0.0) for i in yy.index[low]]
    yc = yy.mask(low); na = yc.isna(); gid = (na != na.shift()).cumsum(); glen = na.groupby(gid).sum(); gfirst = na.groupby(gid).first()
    glen = glen[gfirst]; n_gaps = int(len(glen)); longest = int(glen.max()) if n_gaps else 0; n_long = int((glen > 6).sum())
    yc = yc.interpolate(limit=6)
    start = yc.index[0].ceil("D"); end = yc.index[-1] if yc.index[-1].hour == 23 else yc.index[-1].floor("D") - pd.Timedelta(hours=1)
    yc = yc[start:end]
    log = dict(rows_raw=n_raw, dup=n_dup, start=str(yc.index[0].date()), end=str(yc.index[-1].date()), hours=int(len(yc)), days=int(len(yc) // 24),
               missing_hours=n_missing, anomalies=anomalies, n_anomalies=len(anomalies), borderline=int(border.sum()), n_gaps=n_gaps, longest_gap=longest,
               gaps_over_6h=n_long, border_days={str(k): int(v) for k, v in border[border].groupby(border[border].index.date).size().sort_values(ascending=False).head(4).items()}, nan_left=int(yc.isna().sum()), mean=float(yc.mean()), mn=float(yc.min()), mx=float(yc.max()), raw_min=float(s.min()))
    return yc, log


In [5]:
# ---- Audit dan pra-pemrosesan seluruh wilayah ----
CLEAN = {}
for code in REGIONS:
    CLEAN[code], lg = clean_region(code); RES2["regions"][code] = dict(audit=lg)
aud = pd.DataFrame({c: {k: v for k, v in RES2["regions"][c]["audit"].items() if k != "anomalies"} for c in REGIONS}).T
print(aud[["rows_raw", "start", "end", "dup", "missing_hours", "n_gaps", "longest_gap", "n_anomalies", "borderline", "nan_left", "days", "mean"]].round(0).to_string())
for c in REGIONS:
    for a in RES2["regions"][c]["audit"]["anomalies"][:5]: print(f"  {c}: nilai tak masuk akal dibuang -> {a['t']}  {a['nilai']:.0f} MW (rasio terhadap median lokal {a['rasio']:.2f})")


       rows_raw       start         end dup missing_hours n_gaps longest_gap n_anomalies borderline nan_left  days          mean
PJME     145366  2002-01-02  2018-08-02   4            30     30           1           0          0        0  6057  32079.065207
AEP      121273  2004-10-02  2018-08-02   4            27     27           1           0          0        0  5053  15499.387014
COMED     66497  2011-01-02  2018-08-02   4            11     11           1           0          0        0  2770  11420.266223
DAYTON   121275  2004-10-02  2018-08-02   4            25     25           1           0          1        0  5053   2037.792462
DEOK      57739  2012-01-02  2018-08-02   4             9      9           1           0         30        0  2405   3105.240696
DOM      116189  2005-05-02  2018-08-02   4            23     24           1           1          0        0  4841  10949.575987
DUQ      119068  2005-01-02  2018-08-02   4            24     24           1           0         

*Tabel 3. Hasil pra-pemrosesan data setiap wilayah.*

| Wilayah | Periode | Hari | *Missing hour* | *Duplicate timestamp* | *Removed value* | *Suspicious value* |
|---|---|---:|---:|---:|---:|---:|
| PJME | 2002-01 s.d. 2018-08 | 6.057 | 30 | 4 | 0 | 0 |
| AEP | 2004-10 s.d. 2018-08 | 5.053 | 27 | 4 | 0 | 0 |
| COMED | 2011-01 s.d. 2018-08 | 2.770 | 11 | 4 | 0 | 0 |
| DAYTON | 2004-10 s.d. 2018-08 | 5.053 | 25 | 4 | 0 | 1 |
| DEOK | 2012-01 s.d. 2018-08 | 2.405 | 9 | 4 | 0 | 30 |
| DOM | 2005-05 s.d. 2018-08 | 4.841 | 23 | 4 | 1 | 0 |
| DUQ | 2005-01 s.d. 2018-08 | 4.961 | 24 | 4 | 0 | 0 |
| EKPC | 2013-06 s.d. 2018-08 | 1.888 | 6 | 4 | 0 | 10 |
| FE | 2011-06 s.d. 2018-08 | 2.619 | 10 | 4 | 1 | 0 |
| PJMW | 2002-04 s.d. 2018-08 | 5.967 | 30 | 4 | 1 | 0 |

Berdasarkan tabel di atas, setiap wilayah hanya memiliki *gap* data sepanjang 1 jam (*gap* terpanjang 1 jam) dan 4 *duplicate timestamp*. Jumlah *missing hour* tergolong sangat kecil dibanding jumlah data sehingga *interpolation* tidak mengubah pola beban secara berarti. *Outlier* yang dibuang hanya 3 titik, yaitu DOM pada 2009-12-12 00:00 (1.253 MW, 14% dari *local median*); FE pada 2011-06-01 01:00 (0 MW, 0% dari *local median*); PJMW pada 2003-05-29 00:00 (487 MW, 10% dari *local median*). *Suspicious value* yang dipertahankan terdapat pada DAYTON (1 jam), DEOK (30 jam), EKPC (10 jam). Pada DEOK, 29 dari 30 jam tersebut berada pada 22–25 April 2018 (21 jam pada 22 April) sehingga masuk ke periode *testing* dan dapat sedikit menaikkan *error* DEOK. Pada EKPC, 10 jam tersebut berada pada tahun 2013 (periode *training*). Pemeriksaan ini hanya berdasarkan aturan, tidak diverifikasi ke sumber asli.


#### 2.2.3 Pembentukan Feature dan Target
*Target* yang diprediksi adalah beban pada jam *t* (MW). Prediksi dibuat pukul 00:00 hari *D* untuk 24 jam hari *D*, sehingga hanya data sampai pukul 23:00 hari *D-1* yang boleh dipakai. Oleh karena itu, setiap *feature* berbasis beban memakai *lag* minimal 24 jam atau nilai *snapshot* pada pukul 23:00 hari *D-1*. Aturan ini mencegah *data leakage*, yaitu masuknya informasi dari waktu yang sedang diprediksi ke dalam *feature*. Seluruh 21 *feature* dirinci pada Tabel 4 berikut.

*Tabel 4. Daftar feature yang digunakan.*

| Kelompok | Nama *feature* | Keterangan |
|---|---|---|
| Kalender (6) | `hour`, `dow`, `month` | Jam (0–23), hari dalam pekan (0 = Senin), dan bulan |
| | `is_hol`, `hol_prev`, `hol_next` | Penanda hari libur nasional AS pada hari D, hari sebelumnya, dan hari sesudahnya (*library* `holidays`) |
| *Lag* (6) | `lag24`, `lag48`, `lag168`, `lag336` | Beban pada jam yang sama 1, 2, 7, dan 14 hari sebelumnya |
| | `wk_mean` | Rata-rata beban pada jam yang sama selama 7 hari terakhir |
| | `dmean_prev` | Rata-rata beban 24 jam yang berakhir pada jam t−24 |
| *Snapshot* D-1 (9) | `last_obs`, `last_obs_m3`, `last_obs_m6` | Beban pukul 23:00, 20:00, dan 17:00 pada hari D-1 |
| | `d1_mean`, `d1_max`, `d1_min` | Rata-rata, maksimum, dan minimum beban 24 jam yang berakhir pukul 23:00 hari D-1 |
| | `w1_mean`, `d1_trend` | Rata-rata beban 7 hari yang berakhir pukul 23:00 hari D-1, dan selisih `d1_mean` dengan rata-rata 24 jam sebelumnya |
| | `lag72` | Beban pada jam yang sama 3 hari sebelumnya |


In [6]:
# ---- Pembentukan feature dan target untuk prediksi day-ahead ----
# Skema: prediksi dibuat pukul 00:00 hari D untuk 24 jam hari D (day-ahead). Hanya data sampai 23:00 hari D-1 yang boleh dipakai.
# Setiap feature berbasis beban memakai lag >= 24 jam, atau nilai snapshot pada pukul 23:00 D-1.
def build_features(y):
    d = pd.DataFrame({"y": y}); ix = d.index
    d["hour"] = ix.hour; d["dow"] = ix.dayofweek; d["month"] = ix.month
    hol = holidays.US(years=range(2001, 2020)); dt = pd.Series(ix.date, index=ix)
    d["is_hol"] = dt.map(lambda x: int(x in hol)).values
    d["hol_prev"] = dt.map(lambda x: int((x - pd.Timedelta(days=1)) in hol)).values
    d["hol_next"] = dt.map(lambda x: int((x + pd.Timedelta(days=1)) in hol)).values
    for L in (24, 48, 72, 168, 336): d[f"lag{L}"] = d.y.shift(L)
    d["wk_mean"] = pd.concat([d.y.shift(24 * k) for k in range(1, 8)], axis=1).mean(axis=1)    # rata-rata jam yang sama 7 hari terakhir
    d["dmean_prev"] = d.y.shift(24).rolling(24).mean()
    # feature snapshot D-1: nilai pada 23:00 D-1 (indeks t-(hour+1)) dan statistik 24 jam / 7 hari yang berakhir di titik itu
    yv = d.y.values; n = len(d); pos = np.arange(n) - (d.hour.values + 1); ok = pos >= 0; p = np.clip(pos, 0, None)
    r24 = d.y.rolling(24).mean().values; r24max = d.y.rolling(24).max().values; r24min = d.y.rolling(24).min().values; r168 = d.y.rolling(168).mean().values
    for nm, off in (("last_obs", 0), ("last_obs_m3", 3), ("last_obs_m6", 6)):
        q = pos - off; d[nm] = np.where(q >= 0, yv[np.clip(q, 0, None)], np.nan)
    d["d1_mean"] = np.where(ok, r24[p], np.nan); d["d1_max"] = np.where(ok, r24max[p], np.nan); d["d1_min"] = np.where(ok, r24min[p], np.nan)
    d["w1_mean"] = np.where(ok, r168[p], np.nan); d["d1_trend"] = d.d1_mean - np.where(ok, np.roll(r24, 24)[p], np.nan)
    return d.dropna()

d = build_features(y)
CAL = ["hour", "dow", "month", "is_hol", "hol_prev", "hol_next"]
LAG = ["lag24", "lag48", "lag168", "lag336", "dmean_prev", "wk_mean"]
ISSUE = ["last_obs", "last_obs_m3", "last_obs_m6", "d1_mean", "d1_max", "d1_min", "w1_mean", "d1_trend", "lag72"]
SETS = {"A. Kalender saja": CAL, "B. Kalender + lag >=24 jam": CAL + LAG, "C. B + fitur saat-terbit (D-1)": CAL + LAG + ISSUE}
print("jumlah feature pada set final:", len(SETS["C. B + fitur saat-terbit (D-1)"]), "| baris data setelah feature dibentuk:", len(d))


jumlah feature pada set final: 21 | baris data setelah feature dibentuk: 145032


#### 2.2.4 Pembagian Data dan Uji Data Leakage
Data dibagi secara kronologis (*chronological split*) tanpa *shuffling*. Data *training* mencakup 2002-01-16 sampai 2014-12-31 (113.592 jam), sedangkan data *testing* mencakup 2015-01-01 sampai 2018-08-02 (31.440 jam, 21,7% dari seluruh data). Pembagian kronologis dipilih karena pada kondisi nyata model selalu memprediksi masa depan dari data masa lalu. *Hyperparameter tuning* hanya dilakukan di dalam data *training*, dan *scaling* serta *encoding* hanya di-*fit* pada data *training*. Untuk analisis sepuluh wilayah, periode *testing* disamakan, yaitu 2016-01-01 sampai 2018-08-02, agar hasil antarwilayah dapat dibandingkan.


In [7]:
# ---- Split kronologis (tanpa shuffling) ----
SPLIT = "2015-01-01"
tr, te = d[d.index < SPLIT], d[d.index >= SPLIT]
print(f"train: {tr.index.min().date()} -> {tr.index.max().date()} ({len(tr):,} jam)")
print(f"test : {te.index.min().date()} -> {te.index.max().date()} ({len(te):,} jam) | porsi test {len(te)/len(d):.1%}")
RES["split"] = dict(n_train=int(len(tr)), n_test=int(len(te)), test_share=float(len(te)/len(d)), train_start=str(tr.index.min().date()), train_end=str(tr.index.max().date()),
                    test_start=str(te.index.min().date()), test_end=str(te.index.max().date()), test_mean=float(te.y.mean()), test_std=float(te.y.std()), n_features=len(SETS["C. B + fitur saat-terbit (D-1)"]))
def metrics(t, p):
    mae = mean_absolute_error(t, p); rm = mean_squared_error(t, p) ** .5
    return dict(MAE=float(mae), RMSE=float(rm), R2=float(r2_score(t, p)), rel=float(100 * mae / np.mean(t)))


train: 2002-01-16 -> 2014-12-31 (113,592 jam)
test : 2015-01-01 -> 2018-08-02 (31,440 jam) | porsi test 21.7%


*Data leakage* diuji dengan *perturbation test*. Seluruh nilai beban pada satu hari *D* diganti dengan angka acak, kemudian semua 21 *feature* dihitung ulang. Jika ada *feature* pada jam-jam hari *D* yang ikut berubah, berarti *feature* tersebut memuat informasi dari hari yang sedang diprediksi. Pada 25 hari acak, tidak ada satu pun *feature* yang berubah. Hasil yang sama diperoleh pada 9 wilayah lain (10 hari acak per wilayah). Dengan demikian, rancangan *feature* memenuhi aturan bahwa hanya data sampai pukul 23:00 hari *D-1* yang digunakan.


In [8]:
# ---- Uji data leakage (perturbation test): ganti SELURUH nilai beban hari D dengan angka acak, hitung ulang semua feature,
# lalu cek apakah feature pada jam-jam hari D berubah. Jika ya, feature itu memakai informasi dari hari yang sedang diprediksi (leakage).
FINAL = SETS["C. B + fitur saat-terbit (D-1)"]
rng_l = np.random.RandomState(0)
uniq_days = pd.to_datetime(d.index.date).unique()[400:-5]
test_days = rng_l.choice(uniq_days, 25, replace=False)
changed = {}
for day in test_days:
    yp = y.copy(); msk = (yp.index >= day) & (yp.index < day + pd.Timedelta(days=1))
    yp[msk] = rng_l.uniform(10000, 60000, msk.sum())
    d_pert = build_features(yp)
    rows_ = d.index[(d.index >= day) & (d.index < day + pd.Timedelta(days=1))]
    diff = (d.loc[rows_, FINAL] - d_pert.loc[rows_, FINAL]).abs().max()
    for c_ in diff[diff > 1e-9].index: changed[c_] = changed.get(c_, 0) + 1
print("feature yang berubah saat nilai hari D diganti angka acak:", changed if changed else "TIDAK ADA -> tidak ada data leakage dari hari yang diprediksi (25 hari acak diuji)")
RES["leak_test"] = dict(n_days=25, n_features=len(FINAL), changed_features=changed)


feature yang berubah saat nilai hari D diganti angka acak: TIDAK ADA -> tidak ada data leakage dari hari yang diprediksi (25 hari acak diuji)


## BAB 3. HASIL DAN PEMBAHASAN
### 3.1 Implementation
#### 3.1.1 Alur Implementasi
Alur sistem proyek dilakukan berdasarkan *workflow* yang ditunjukkan diagram alir pada Gambar 1 sebagai berikut.

![Gambar 1. Diagram alir implementasi model AI](report/gambar_alur_implementasi.png)

*Gambar 1. Diagram alir implementasi model AI.*


#### 3.1.2 Regression Day-Ahead
Model utama adalah *HistGradientBoostingRegressor* dari scikit-learn. Model ini membangun banyak *decision tree* secara berurutan, dengan setiap *tree* memperbaiki sisa *error* dari *tree* sebelumnya, dan memakai histogram untuk mempercepat pencarian titik pemisah. Model ini pun dilakukan *tuning* dengan *hyperparameter tuning* menggunakan *GridSearchCV* pada 16 kombinasi parameter, *cross-validation* berbasis *TimeSeriesSplit* dengan 4 *fold*, dan skor MAE. Seluruh proses ini hanya memakai data *training*. Parameter terbaik adalah *learning_rate* 0,06, *max_iter* 800, *max_leaf_nodes* 31, dan *min_samples_leaf* 50.


In [9]:
# ---- Hyperparameter tuning HistGradientBoosting (HGB): GridSearchCV + TimeSeriesSplit(4) HANYA pada data train ----
GRID = {"learning_rate": [0.03, 0.06], "max_iter": [400, 800], "max_leaf_nodes": [31, 63], "min_samples_leaf": [50, 100]}
FINAL = SETS["C. B + fitur saat-terbit (D-1)"]
t0 = time.time()
gs = GridSearchCV(HistGradientBoostingRegressor(random_state=SEED), GRID, cv=TimeSeriesSplit(4), scoring="neg_mean_absolute_error", n_jobs=-1).fit(tr[FINAL], tr.y)
print("parameter terbaik:", gs.best_params_, "| MAE cross-validation (train):", round(-gs.best_score_, 1), f"| {time.time()-t0:.0f}s")
BP = gs.best_params_; RES["tuning"] = dict(best=BP, cv_mae=float(-gs.best_score_), grid_size=int(np.prod([len(v) for v in GRID.values()])))


parameter terbaik: {'learning_rate': 0.06, 'max_iter': 800, 'max_leaf_nodes': 31, 'min_samples_leaf': 50} | MAE cross-validation (train): 1161.7 | 194s


Hasil yang diperoleh dibandingkan dengan pembanding, yaitu Random Forest (300 *tree*, *min_samples_leaf* 5, *max_features* 0,5) dan Ridge Regression (alpha 10) dengan *one-hot encoding* untuk *feature* kalender. Empat *baseline* tanpa proses *learning* juga dihitung (sel berikut): (1) persistensi harian (lag24), yaitu beban pada jam yang sama kemarin; (2) persistensi mingguan (lag168), yaitu beban pada jam yang sama minggu lalu; (3) rata-rata lag24, lag48, dan lag168; serta (4) rata-rata beban pada jam yang sama selama 7 hari terakhir. Model harus mengalahkan *baseline* terbaik agar dianggap memiliki kemampuan prediksi.


In [10]:
# ---- Baseline sederhana (tanpa belajar) ----
T = te.y.values
base = {"Persistensi harian (lag24)": te.lag24.values,
        "Persistensi mingguan (lag168)": te.lag168.values,
        "Rata-rata lag24/48/168": te[["lag24", "lag48", "lag168"]].mean(axis=1).values,
        "Rata-rata jam sama 7 hari": te.wk_mean.values}
RES["baseline"] = {k: metrics(T, v) for k, v in base.items()}
print(pd.DataFrame(RES["baseline"]).T.round(3).to_string())
BEST_BASE = min(RES["baseline"], key=lambda k: RES["baseline"][k]["MAE"]); print("\nbaseline terbaik:", BEST_BASE)


                                    MAE      RMSE     R2     rel
Persistensi harian (lag24)     2209.047  3025.216  0.780   7.053
Persistensi mingguan (lag168)  3400.302  4645.376  0.481  10.856
Rata-rata lag24/48/168         2372.253  3145.625  0.762   7.574
Rata-rata jam sama 7 hari      2710.059  3612.987  0.686   8.652

baseline terbaik: Persistensi harian (lag24)


#### 3.1.3 Clustering Profil Beban Harian
Pada proyek ini, setiap hari direpresentasikan oleh 24 nilai beban per jam (6.057 hari lengkap pada PJME). Analisis tersebut bersifat deskriptif karena memakai data hari itu sendiri sehingga tidak ada isu *data leakage*. Sementara itu, konfigurasi awal memakai data yang distandarkan per jam dan K-Means dengan k dipilih dari Silhouette Score. Konfigurasi tersebut kemudian dilakukan *tuning* dan dibandingkan dengan algoritma lain. Rancangan konfigurasi tersebut dirinci sebagai berikut.
1. ***Preprocessing* (10 varian):** standardisasi per jam (konfigurasi awal), MW asli tanpa standardisasi, *robust scaling*, *min-max*, log lalu standardisasi, bentuk saja (*z-score* per hari), bentuk saja (rasio ke rata-rata harian), bentuk + level (bentuk ditambah rata-rata harian terstandar), PCA 95% variansi, dan PCA 3 komponen.
2. **Algoritma (6 jenis):** K-Means, *Gaussian Mixture Model* (kovarians *full* dan *diagonal*), *Agglomerative clustering* (*linkage* Ward dan *average*), serta *spectral clustering*. Jumlah *cluster* k diuji dari 2 sampai 6, sehingga totalnya 265 konfigurasi pada PJME.
3. **Ukuran:** Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz, stabilitas, dan R² rekonstruksi. Stabilitas adalah rata-rata *Adjusted Rand Index* (ARI) antara dua hasil *clustering* pada dua *subsample* acak 70% data, dibandingkan pada hari yang sama (5 ulangan). R² rekonstruksi menunjukkan seberapa besar variasi profil MW yang dijelaskan oleh rata-rata *cluster*.

Silhouette Score dihitung di ruang *feature* milik tiap konfigurasi, sehingga angkanya tidak bisa dibandingkan langsung antar-*preprocessing* karena jarak diukur di ruang yang berbeda. Untuk perbandingan, setiap hasil *clustering* dinilai ulang di dua ruang pembanding netral, yaitu S1 (data distandarkan per jam) dan S2 (bentuk profil saja, *z-score* per hari), dan *score* didefinisikan sebagai rata-rata Silhouette di S1 dan S2. Sebelum hasil dilihat, ditetapkan aturan seleksi: pilih konfigurasi dengan *score* tertinggi yang memiliki stabilitas minimal 0,8 dan *cluster* terkecil minimal 5% dari seluruh hari. Perbedaan antar-kandidat teratas diuji dengan 10 kali *subsample* 70% berpasangan (kandidat dan K-Means standar dinilai pada *subsample* yang sama). Pengujian diulang pada 9 wilayah lain dengan 5 konfigurasi terpilih.


In [11]:
# Satu baris = satu hari; 24 kolom = beban tiap jam (MW). Clustering bersifat DESKRIPTIF (memakai data hari itu sendiri).
daily = y.groupby(y.index.date).apply(lambda s: s.values)
M = np.vstack(daily.values); dates = pd.to_datetime(daily.index)
print("matriks profil harian:", M.shape)


matriks profil harian: (6057, 24)


In [12]:
# ---- Fungsi bantu untuk tuning clustering: pra-pemrosesan x algoritma x k ----
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.cluster import AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import calinski_harabasz_score, adjusted_rand_score, normalized_mutual_info_score, pairwise_distances

def cluster_preps(M):
    """Variasi pra-pemrosesan profil harian (24 nilai per hari)."""
    P = {}
    P["raw_MW"] = M.copy()                                                  # tanpa penskalaan (satuan MW)
    P["std_per_jam (awal)"] = StandardScaler().fit_transform(M)             # konfigurasi awal proyek
    P["robust_per_jam"] = RobustScaler().fit_transform(M)
    P["minmax_per_jam"] = MinMaxScaler().fit_transform(M)
    P["log_lalu_std"] = StandardScaler().fit_transform(np.log(M))
    mu = M.mean(1, keepdims=True); sd = M.std(1, keepdims=True)
    P["bentuk_z_per_hari"] = (M - mu) / sd                                  # hanya bentuk profil (level dibuang)
    P["bentuk_rasio_rata2"] = M / mu
    P["bentuk_z + level"] = np.hstack([P["bentuk_z_per_hari"], StandardScaler().fit_transform(mu) * 2.0])
    P["pca_95%_var"] = PCA(0.95, random_state=SEED).fit_transform(P["std_per_jam (awal)"])
    P["pca_3_komponen"] = PCA(3, random_state=SEED).fit_transform(P["std_per_jam (awal)"])
    return P

def cluster_make(alg, k, seed=SEED):
    if alg == "kmeans": return KMeans(k, n_init=10, random_state=seed)
    if alg == "gmm_full": return GaussianMixture(k, covariance_type="full", n_init=3, random_state=seed, reg_covar=1e-4)
    if alg == "gmm_diag": return GaussianMixture(k, covariance_type="diag", n_init=3, random_state=seed)
    if alg == "agglo_ward": return AgglomerativeClustering(k, linkage="ward")
    if alg == "agglo_average": return AgglomerativeClustering(k, linkage="average")
    if alg == "spectral": return SpectralClustering(k, affinity="nearest_neighbors", n_neighbors=15, random_state=seed, assign_labels="kmeans")
    raise ValueError(alg)

def cluster_stability(alg, k, X, n_rep=5):
    """Stabilitas = rata-rata Adjusted Rand Index antara dua model yang dilatih pada dua subsampel 70% berbeda (dibandingkan pada hari yang sama)."""
    rs = np.random.RandomState(0); n = len(X); out = []
    for r in range(n_rep):
        a = rs.choice(n, int(0.7 * n), replace=False); b = rs.choice(n, int(0.7 * n), replace=False); common = np.intersect1d(a, b)
        la = cluster_make(alg, k, r).fit_predict(X[a]); lb = cluster_make(alg, k, r + 100).fit_predict(X[b])
        ia = {j: i for i, j in enumerate(a)}; ib = {j: i for i, j in enumerate(b)}
        out.append(adjusted_rand_score([la[ia[j]] for j in common], [lb[ib[j]] for j in common]))
    return float(np.mean(out))

def cluster_recon(M, lab):
    """R2 rekonstruksi: seberapa besar variasi profil harian (dalam MW) yang dijelaskan oleh profil rata-rata tiap cluster."""
    mu = M.mean(0); sst = ((M - mu) ** 2).sum(); sse = sum(((M[lab == c] - M[lab == c].mean(0)) ** 2).sum() for c in np.unique(lab))
    return float(1 - sse / sst), float(np.sqrt(sse / M.size))

def cluster_grid(M, dts, P, plan, ks=(2, 3, 4, 5, 6), verbose=True):
    """plan = daftar (nama_prep, algoritma, daftar_k). Metrik:
       sil_S1 / sil_S2 = Silhouette pada dua ruang pembanding netral (S1 = profil distandarkan per jam, S2 = bentuk profil per hari);
       score = rata-rata sil_S1 dan sil_S2; stab = stabilitas (ARI); recon_R2 = R2 rekonstruksi profil dalam MW."""
    n = len(M); D1 = pairwise_distances(P["std_per_jam (awal)"]).astype(np.float32); D2 = pairwise_distances(P["bentuk_z_per_hari"]).astype(np.float32)
    wk = (dts.dayofweek >= 5).astype(int); mo = dts.month.values; rows = []; t0 = time.time()
    for pn, alg, kk in plan:
        X = P[pn]; Dm = pairwise_distances(X).astype(np.float32)
        for k in kk:
            lab = cluster_make(alg, k).fit_predict(X)
            if len(np.unique(lab)) < 2: continue
            s1 = silhouette_score(D1, lab, metric="precomputed"); s2 = silhouette_score(D2, lab, metric="precomputed"); r2, rm = cluster_recon(M, lab)
            rows.append(dict(prep=pn, alg=alg, k=k, sil_own=float(silhouette_score(Dm, lab, metric="precomputed")), sil_S1=float(s1), sil_S2=float(s2), score=float((s1 + s2) / 2),
                             db=float(davies_bouldin_score(X, lab)), ch=float(calinski_harabasz_score(X, lab)), stab=cluster_stability(alg, k, X), min_frac=float(np.bincount(lab).min() / n),
                             recon_R2=r2, recon_RMSE_MW=rm, nmi_month=float(normalized_mutual_info_score(mo, lab)), nmi_weekend=float(normalized_mutual_info_score(wk, lab))))
        if verbose: print(f"{pn:22s} {alg:14s} selesai ({time.time() - t0:.0f} dtk)")
    return pd.DataFrame(rows)

def cluster_final(M, ks=range(2, 7)):
    """Konfigurasi terpilih: K-Means pada profil dalam MW (tanpa standardisasi); k dengan Silhouette tertinggi.
       Konfigurasi awal (K-Means + standardisasi per jam) disimpan sebagai pembanding."""
    Xstd = StandardScaler().fit_transform(M); srows = []
    for k in ks:
        lb = KMeans(k, n_init=10, random_state=SEED).fit_predict(M); lb0 = KMeans(k, n_init=10, random_state=SEED).fit_predict(Xstd)
        srows.append(dict(k=k, silhouette=float(silhouette_score(M, lb)), davies_bouldin=float(davies_bouldin_score(M, lb)), min_cluster=int(np.bincount(lb).min()),
                          silhouette_std=float(silhouette_score(Xstd, lb0)), ari_vs_std=float(adjusted_rand_score(lb, lb0))))
    kb = max(srows, key=lambda r: r["silhouette"])["k"]; lab = KMeans(kb, n_init=10, random_state=SEED).fit_predict(M)
    order = np.argsort([M[lab == c].mean() for c in range(kb)]); rm = {int(o): i for i, o in enumerate(order)}
    return srows, kb, np.array([rm[int(l)] for l in lab])


In [13]:
# ---- Tuning clustering (grid ringkas dijalankan di sini) ----
# Pra-pemrosesan: 10 variasi (K-Means) | Algoritma lain pada data standar: GMM diagonal, Agglomerative Ward (k = 2..6), Spectral (k = 2, 3)
CP = cluster_preps(M)
plan = [(pn, "kmeans", (2, 3, 4, 5, 6)) for pn in CP] + [("std_per_jam (awal)", "gmm_diag", (2, 3, 4, 5, 6)), ("std_per_jam (awal)", "agglo_ward", (2, 3, 4, 5, 6)), ("std_per_jam (awal)", "spectral", (2, 3))]
ct = cluster_grid(M, dates, CP, plan)
ok = ct[(ct.stab >= 0.8) & (ct.min_frac >= 0.05)]
best_k = ct.loc[ct.groupby(["prep", "alg"]).score.idxmax()].sort_values("score", ascending=False)
show = ["prep", "alg", "k", "score", "sil_own", "db", "stab", "min_frac", "recon_R2"]
print("\nKonfigurasi terbaik per (pra-pemrosesan, algoritma), diurutkan menurut skor ruang netral:")
print(best_k[show].round(3).to_string(index=False))
RES["cluster_tuning"] = dict(grid_ringkas=ct.round(4).to_dict("records"), n_eligible=int(len(ok)), n_total=int(len(ct)))


raw_MW                 kmeans         selesai (6 dtk)


std_per_jam (awal)     kmeans         selesai (12 dtk)


robust_per_jam         kmeans         selesai (19 dtk)


minmax_per_jam         kmeans         selesai (25 dtk)


log_lalu_std           kmeans         selesai (31 dtk)


...

pca_95%_var            kmeans         selesai (55 dtk)


pca_3_komponen         kmeans         selesai (60 dtk)


std_per_jam (awal)     gmm_diag       selesai (73 dtk)


std_per_jam (awal)     agglo_ward     selesai (108 dtk)


std_per_jam (awal)     spectral       selesai (127 dtk)

Konfigurasi terbaik per (pra-pemrosesan, algoritma), diurutkan menurut skor ruang netral:
              prep        alg  k  score  sil_own    db  stab  min_frac  recon_R2
std_per_jam (awal)   spectral  2  0.282    0.472 0.821 0.844     0.327     0.568
  bentuk_z + level     kmeans  2  0.279    0.335 1.183 0.979     0.332     0.552
            raw_MW     kmeans  2  0.276    0.494 0.780 0.988     0.344     0.573
std_per_jam (awal) agglo_ward  2  0.273    0.482 0.797 0.537     0.264     0.520
    minmax_per_jam     kmeans  2  0.273    0.480 0.807 0.989     0.355     0.570
    robust_per_jam     kmeans  2  0.273    0.480 0.808 0.990     0.352     0.568
std_per_jam (awal)     kmeans  2  0.271    0.477 0.809 0.972     0.363     0.562
    pca_3_komponen     kmeans  2  0.271    0.489 0.784 0.977     0.364     0.561
       pca_95%_var     kmeans  2  0.271    0.489 0.784 0.977     0.364     0.561
      log_lalu_std     kmeans  2  0.260    

Sel di atas menjalankan **grid ringkas** di dalam notebook (semua *preprocessing* dengan K-Means, plus GMM diagonal, Ward, dan *spectral* pada data standar). Grid penuh (265 konfigurasi) dan pengujian pada 9 wilayah lain dijalankan lewat skrip terpisah `cluster_tune.py`, `cluster_pairs.py`, dan `cluster_regions.py` di folder `cluster_tuning/` karena memakan waktu lama. Hasilnya (`cluster_tune_*.json`) dimuat dan dibahas pada Subbab 3.2.7.

#### 3.1.4 Evaluasi Tambahan
Empat analisis tambahan dilakukan di luar evaluasi pada satu periode *testing*. *Ablation study* membandingkan tiga kelompok *feature* (kalender saja; kalender dan *lag*; semua *feature*) untuk melihat kontribusi tiap kelompok. Evaluasi *rolling-origin* mengulang proses *training* dan *testing* pada 8 periode berurutan (masing-masing sekitar 6 bulan) untuk melihat konsistensi hasil terhadap waktu. Analisis *error* melihat sebaran *error* menurut jam, bulan, dan jenis hari. *Permutation importance* mengukur seberapa besar MAE naik ketika nilai suatu *feature* diacak. Keempatnya dijalankan pada Subbab 3.2.3 sampai 3.2.5.

#### 3.1.5 Pipeline Multi-Wilayah
Setiap wilayah dilatih dengan model tersendiri menggunakan *pipeline* yang sama, yaitu 21 *feature* yang sama, empat *baseline*, *tuning* HGB pada grid 8 kombinasi dengan *TimeSeriesSplit* 4 *fold*, pembanding Ridge, evaluasi *rolling-origin*, *clustering* K-Means, dan *perturbation test*. Terdapat beberapa perbedaan dengan analisis utama PJME.
1. Periode *testing* disamakan pada 2016-01-01 sampai 2018-08-02 untuk semua wilayah. Wilayah dengan data pendek seperti EKPC (mulai pertengahan 2013) tetap memiliki data *training* sekitar 2,6 tahun. PJME diulang pada protokol ini hanya untuk tabel perbandingan, sedangkan hasil utama PJME (*testing* mulai 2015) tidak diubah. Karena itu, angka PJME pada tabel perbandingan sedikit berbeda dari hasil utama.
2. Set *feature* dan rentang *hyperparameter* tidak disetel ulang untuk tiap wilayah karena keduanya dirancang pada PJME. Untuk wilayah lain, hanya nilai *hyperparameter* yang dicari ulang pada data *training*.
3. Untuk menghemat waktu komputasi, Random Forest tidak dijalankan per wilayah, *ablation study* hanya memakai tiga set pada satu pembagian data *holdout*, *perturbation test* memakai 10 hari acak, dan *permutation importance* dihitung pada 4.000 jam.

Prediksi *testing* dan model 21 *feature* tiap wilayah disimpan ke `results/regions_out/` (berkas besar, tidak ikut di repo; dibutuhkan hanya untuk dashboard web). Sel berikut mendefinisikan fungsi *pipeline*; pemanggilannya ada pada Subbab 3.2.6.


In [14]:
def cluster_region(y):
    cnt = y.groupby(y.index.normalize()).count(); good = cnt[cnt == 24].index; Mx = y[y.index.normalize().isin(good)].values.reshape(-1, 24)
    srows, kb, lab_ = cluster_final(Mx)
    return srows, kb, lab_, Mx, pd.to_datetime(good)

def run_region(code, y, split=SPLIT_MR, n_leak=10, export=True):
    t0 = time.time(); R_ = {}; FIN = SETS["C. B + fitur saat-terbit (D-1)"]
    d_ = build_features(y); tr_, te_ = d_[d_.index < split], d_[d_.index >= split]; T_ = te_.y.values
    R_["split"] = dict(n_train=int(len(tr_)), n_test=int(len(te_)), train_start=str(tr_.index.min().date()), train_end=str(tr_.index.max().date()),
                       test_start=str(te_.index.min().date()), test_end=str(te_.index.max().date()), test_mean=float(te_.y.mean()), test_std=float(te_.y.std()))
    # baseline
    bs = {"Persistensi harian (lag24)": te_.lag24.values, "Persistensi mingguan (lag168)": te_.lag168.values,
          "Rata-rata lag24/48/168": te_[["lag24", "lag48", "lag168"]].mean(axis=1).values, "Rata-rata jam sama 7 hari": te_.wk_mean.values}
    R_["baseline"] = {k: metrics(T_, v) for k, v in bs.items()}; bb = min(R_["baseline"], key=lambda k: R_["baseline"][k]["MAE"]); R_["best_baseline"] = bb
    # tuning: hanya di train, TimeSeriesSplit(4)
    GRID_ = {"learning_rate": [0.06], "max_iter": [400, 800], "max_leaf_nodes": [31, 63], "min_samples_leaf": [50, 100]}
    gs_ = GridSearchCV(HistGradientBoostingRegressor(random_state=SEED), GRID_, cv=TimeSeriesSplit(4), scoring="neg_mean_absolute_error", n_jobs=-1).fit(tr_[FIN], tr_.y)
    bp = gs_.best_params_; R_["tuning"] = dict(best=bp, cv_mae=float(-gs_.best_score_), grid_size=8)
    # ablation study (uji holdout, 3 set)
    ab = []
    for nm, cols in SETS.items():
        m_ = HistGradientBoostingRegressor(random_state=SEED, **bp).fit(tr_[cols], tr_.y); ab.append(dict(set=nm, n_fitur=len(cols), **metrics(T_, m_.predict(te_[cols]))))
    R_["ablation"] = ab
    # model
    hgb_ = HistGradientBoostingRegressor(random_state=SEED, **bp).fit(tr_[FIN], tr_.y); ph = hgb_.predict(te_[FIN])
    oh = lambda df: pd.get_dummies(df[FIN].astype({"hour": "category", "dow": "category", "month": "category"}), columns=["hour", "dow", "month"], dtype=float)
    Otr = oh(tr_); Ote = oh(te_).reindex(columns=Otr.columns, fill_value=0); sc_ = StandardScaler().fit(Otr); rg_ = Ridge(alpha=10).fit(sc_.transform(Otr), tr_.y)
    R_["models"] = {"HistGradientBoosting (tuned)": metrics(T_, ph), "Ridge (one-hot kalender)": metrics(T_, rg_.predict(sc_.transform(Ote)))}
    h_ = R_["models"]["HistGradientBoosting (tuned)"]; b_ = R_["baseline"][bb]
    R_["skill"] = dict(mae_reduction_pct=float(100 * (1 - h_["MAE"] / b_["MAE"])), rmse_reduction_pct=float(100 * (1 - h_["RMSE"] / b_["RMSE"])))
    # rolling-origin
    nsp = int(min(8, (len(d_) - 8760) // 4380)); rows_ = []
    for i, (a, b) in enumerate(TimeSeriesSplit(nsp, test_size=4380).split(d_)):
        A_, B_ = d_.iloc[a], d_.iloc[b]; m_ = HistGradientBoostingRegressor(random_state=SEED, **bp).fit(A_[FIN], A_.y); pm_ = m_.predict(B_[FIN]); mu_ = B_.y.mean()
        rows_.append(dict(lipatan=i + 1, uji_mulai=str(B_.index[0].date()), uji_akhir=str(B_.index[-1].date()), rel_HGB=100 * mean_absolute_error(B_.y, pm_) / mu_,
                          rel_lag24=100 * mean_absolute_error(B_.y, B_.lag24) / mu_, R2_HGB=r2_score(B_.y, pm_)))
    ro_ = pd.DataFrame(rows_)
    R_["rolling"] = dict(rel_hgb_mean=float(ro_.rel_HGB.mean()), rel_hgb_std=float(ro_.rel_HGB.std()), rel_lag24_mean=float(ro_.rel_lag24.mean()), r2_mean=float(ro_.R2_HGB.mean()),
                         hgb_wins=int((ro_.rel_HGB < ro_.rel_lag24).sum()), n=int(len(ro_)), rows=ro_.round(3).to_dict("records"))
    # error
    E_ = te_[["y"]].copy(); E_["pred"] = ph; E_["err"] = E_.pred - E_.y; E_["ae"] = E_.err.abs(); E_["pe"] = 100 * E_.ae / E_.y; E_["dow"] = te_.dow; E_["is_hol"] = te_.is_hol; E_["month"] = te_.month; E_["hour"] = te_.hour; E_["date"] = te_.index.date
    dt_ = E_.assign(tipe=np.where(E_.is_hol == 1, "Libur federal", np.where(E_.dow >= 5, "Akhir pekan", "Hari kerja"))).groupby("tipe").pe.mean(); dd_ = E_.groupby("date").agg(mape=("pe", "mean")).sort_values("mape", ascending=False)
    R_["errors"] = dict(by_hour=[float(v) for v in E_.groupby("hour").pe.mean().values], by_month=[float(E_[E_.month == m].pe.mean()) if (E_.month == m).any() else 0.0 for m in range(1, 13)],
                        daytype={k: float(v) for k, v in dt_.items()}, bias=float(E_.err.mean()), p90=float(E_.ae.quantile(.9)), p99=float(E_.ae.quantile(.99)), max=float(E_.ae.max()),
                        worst=[dict(date=str(i), mape=float(r.mape)) for i, r in dd_.head(5).iterrows()], worst10_weekend=int(sum(pd.Timestamp(i).dayofweek >= 5 for i in dd_.head(10).index)))
    # importance
    rs_ = np.random.RandomState(SEED); sub_ = rs_.choice(len(te_), min(4000, len(te_)), replace=False)
    pi_ = permutation_importance(hgb_, te_[FIN].iloc[sub_], te_.y.iloc[sub_], n_repeats=3, random_state=SEED, scoring="neg_mean_absolute_error")
    imp_ = pd.Series(pi_.importances_mean, index=FIN).sort_values(ascending=False); R_["importance"] = {k: float(v) for k, v in imp_.head(10).items()}
    # clustering (hari lengkap saja): K-Means pada profil MW (konfigurasi hasil tuning), lihat cells_cluster.py
    srows, kb, lab_, Mx, dts = cluster_region(y)
    inf_ = pd.DataFrame({"date": dts, "cl": lab_}); inf_["weekend"] = (inf_.date.dt.dayofweek >= 5).astype(int); inf_["mean_mw"] = Mx.mean(1); inf_["peak_h"] = Mx.argmax(1); inf_["month"] = inf_.date.dt.month
    sm_ = inf_.groupby("cl").agg(n_hari=("cl", "size"), rata2_MW=("mean_mw", "mean"), persen_akhir_pekan=("weekend", "mean"), jam_puncak_median=("peak_h", "median")).round(3)
    sk_ = [r for r in srows if r["k"] == kb][0]
    R_["silhouette"] = srows; R_["cluster"] = dict(k=int(kb), silhouette=sk_["silhouette"], silhouette_std=sk_["silhouette_std"], ari_vs_std=sk_["ari_vs_std"], summary=sm_.reset_index().to_dict("records"),
                                                    month_pct=(pd.crosstab(inf_.cl, inf_.month, normalize="index") * 100).round(1).to_dict("index"))
    # uji data leakage (perturbation)
    rl_ = np.random.RandomState(0); ud_ = pd.to_datetime(d_.index.date).unique()[400:-5]; chg = {}
    for day in rl_.choice(ud_, n_leak, replace=False):
        yp_ = y.copy(); mk_ = (yp_.index >= day) & (yp_.index < day + pd.Timedelta(days=1)); yp_[mk_] = rl_.uniform(0.3 * y.mean(), 1.9 * y.mean(), mk_.sum())
        dp_ = build_features(yp_); rw_ = d_.index[(d_.index >= day) & (d_.index < day + pd.Timedelta(days=1))]
        df_ = (d_.loc[rw_, FIN] - dp_.reindex(rw_)[FIN]).abs().max()
        for c_ in df_[df_ > 1e-9].index: chg[c_] = chg.get(c_, 0) + 1
    R_["leak_test"] = dict(n_days=n_leak, n_features=len(FIN), changed_features=chg)
    if export:   # data web + model + fixture
        td_ = pd.DataFrame({"y": te_.y.values, "p": ph, "b": te_.lag24.values}, index=te_.index); days_ = {}
        for dd, g_ in td_.groupby(td_.index.date):
            if len(g_) == 24: days_[str(dd)] = [[int(round(v)) for v in g_[c].values] for c in ("y", "p", "b")]
        cl_ = [dict(name=f"Cluster {c}", n=int((lab_ == c).sum()), mean=[int(round(v)) for v in Mx[lab_ == c].mean(0)], p10=[int(round(v)) for v in np.percentile(Mx[lab_ == c], 10, 0)],
                    p90=[int(round(v)) for v in np.percentile(Mx[lab_ == c], 90, 0)]) for c in range(kb)]
        R_["_web"] = dict(days=days_, clusters=cl_, importance={k: round(float(v), 1) for k, v in imp_.head(10).items()})
        trees = []
        for it in hgb_._predictors:
            nd = it[0].nodes
            trees.append(dict(f=[int(x) if not lf else -1 for x, lf in zip(nd["feature_idx"], nd["is_leaf"])], t=[float(x) for x in nd["num_threshold"]], l=[int(x) for x in nd["left"]],
                              r=[int(x) for x in nd["right"]], v=[float(f"{x:.12g}") for x in nd["value"]], mg=[int(x) for x in nd["missing_go_to_left"]]))
        R_["_model"] = dict(features=FIN, base=float(hgb_._baseline_prediction.ravel()[0]), trees=trees)
        ix_ = np.random.RandomState(1).choice(len(te_), 40, replace=False)
        R_["_fixture"] = dict(X=te_[FIN].iloc[ix_].values.tolist(), y=[float(v) for v in hgb_.predict(te_[FIN].iloc[ix_])])
    R_["seconds"] = round(time.time() - t0)
    return R_


### 3.2 Results
#### 3.2.1 Eksplorasi Data

In [15]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ym = y.resample("MS").mean(); ax[0].plot(ym.index, ym.values, lw=1); ax[0].set_title("Rata-rata beban bulanan (MW)")
tmp = pd.DataFrame({"y": y}); tmp["hour"] = tmp.index.hour; tmp["dow"] = tmp.index.dayofweek; tmp["season"] = tmp.index.month.map(lambda m: "Des-Feb" if m in (12,1,2) else "Jun-Agu" if m in (6,7,8) else "Lainnya")
for s, g in tmp.groupby("season"): ax[1].plot(g.groupby("hour").y.mean(), label=s)
ax[1].set_title("Profil rata-rata per jam menurut musim"); ax[1].set_xlabel("jam"); ax[1].legend()
ax[2].bar(range(7), tmp.groupby("dow").y.mean().values); ax[2].set_xticks(range(7)); ax[2].set_xticklabels(["Sen","Sel","Rab","Kam","Jum","Sab","Min"])
ax[2].set_ylim(25000, 35000); ax[2].set_title("Rata-rata per hari dalam pekan (MW)")
plt.tight_layout(); plt.savefig("results/fig_eda.png", bbox_inches="tight"); plt.show()
RES["eda"] = dict(dow_mean=[float(v) for v in tmp.groupby("dow").y.mean().values])


Pada gambar di atas (kiri), rata-rata beban bulanan naik dan turun secara periodik dengan dua puncak dalam setahun, yaitu pada musim panas dan musim dingin, dalam rentang sekitar 26.300 MW (April 2017) sampai 40.600 MW (Juli 2011). Tinggi puncak berbeda dari tahun ke tahun sehingga bulan saja tidak cukup untuk menentukan beban. Pada panel tengah, profil musim panas (Juni–Agustus) mencapai puncak sekitar 43.500 MW pada pukul 17:00, sedangkan profil musim dingin (Desember–Februari) memiliki dua puncak, yaitu pada pagi hari dan malam hari (puncak tertinggi sekitar 38.100 MW pada pukul 19:00). Dari sudut pandang Teknik Elektro, bentuk ini sesuai dengan dugaan bahwa beban pendingin ruangan mendominasi pada sore hari di musim panas, sedangkan beban pemanas dan penerangan menghasilkan dua puncak pada musim dingin. Dugaan ini belum diuji karena *dataset* tidak memuat data suhu.

Pada panel kanan, rata-rata beban hari Sabtu (30.163 MW) dan Minggu (29.406 MW) lebih rendah sekitar 8,6% dan 10,9% dibanding rata-rata Senin sampai Jumat (32.996 MW). Hal ini menunjukkan bahwa aktivitas industri dan komersial yang lebih rendah pada akhir pekan terlihat langsung pada kurva beban. Berdasarkan ketiga pola tersebut, jam, hari dalam pekan, bulan, dan hari libur dipilih sebagai *feature* kalender.

#### 3.2.2 Hasil Regression

In [16]:
# ---- Perbandingan model dan baseline pada feature final (set C) ----
Xtr, Xte = tr[FINAL], te[FINAL]
models = {}
hgb = HistGradientBoostingRegressor(random_state=SEED, **BP).fit(Xtr, tr.y); models["HistGradientBoosting (tuned)"] = hgb.predict(Xte)
rf = RandomForestRegressor(300, min_samples_leaf=5, max_features=0.5, n_jobs=-1, random_state=SEED).fit(Xtr, tr.y); models["Random Forest"] = rf.predict(Xte)
def onehot(df): return pd.get_dummies(df[FINAL].astype({"hour": "category", "dow": "category", "month": "category"}), columns=["hour", "dow", "month"], dtype=float)
Otr = onehot(tr); Ote = onehot(te).reindex(columns=Otr.columns, fill_value=0); sc = StandardScaler().fit(Otr)
rg = Ridge(alpha=10).fit(sc.transform(Otr), tr.y); models["Ridge (one-hot kalender)"] = rg.predict(sc.transform(Ote))
RES["models"] = {k: metrics(T, v) for k, v in models.items()}
tab = pd.DataFrame({**{f"[baseline] {k}": v for k, v in RES["baseline"].items()}, **RES["models"]}).T
print(tab.round(3).to_string())
best_base = RES["baseline"][BEST_BASE]; h = RES["models"]["HistGradientBoosting (tuned)"]
RES["skill"] = dict(mae_reduction_pct=float(100 * (1 - h["MAE"] / best_base["MAE"])), rmse_reduction_pct=float(100 * (1 - h["RMSE"] / best_base["RMSE"])))
print("\nPenurunan MAE vs baseline terbaik (%s): %.1f%% | RMSE: %.1f%%" % (BEST_BASE, RES["skill"]["mae_reduction_pct"], RES["skill"]["rmse_reduction_pct"]))
pred = models["HistGradientBoosting (tuned)"]


                                               MAE      RMSE     R2     rel
[baseline] Persistensi harian (lag24)     2209.047  3025.216  0.780   7.053
[baseline] Persistensi mingguan (lag168)  3400.302  4645.376  0.481  10.856
[baseline] Rata-rata lag24/48/168         2372.253  3145.625  0.762   7.574
[baseline] Rata-rata jam sama 7 hari      2710.059  3612.987  0.686   8.652
HistGradientBoosting (tuned)              1274.602  1868.668  0.916   4.069
Random Forest                             1370.797  1967.501  0.907   4.376
Ridge (one-hot kalender)                  1404.784  1942.335  0.909   4.485

Penurunan MAE vs baseline terbaik (Persistensi harian (lag24)): 42.3% | RMSE: 38.2%


Pada tabel di atas, *baseline* terbaik adalah persistensi harian (lag24) dengan MAE 2.209 MW dan R² 0,780. Persistensi mingguan (lag168) jauh lebih buruk (MAE 3.400 MW), yang menunjukkan bahwa kondisi satu hari sebelumnya lebih informatif daripada kondisi satu minggu sebelumnya untuk prediksi *day-ahead*. HGB mencapai MAE 1.275 MW, RMSE 1.869 MW, dan R² 0,916, sehingga MAE turun 42,3% dan RMSE turun 38,2% dibanding *baseline* lag24. *Relative error* 4,07% berarti rata-rata penyimpangan prediksi sekitar 4,1% dari beban rata-rata pada periode *testing* (31.323 MW). Random Forest (R² 0,907) dan Ridge (R² 0,909) berada sangat dekat dengan HGB. Selisih antaralgoritma ini jauh lebih kecil daripada selisih terhadap *baseline* sehingga peningkatan akurasi terutama berasal dari *feature* yang digunakan dan bukan dari kompleksitas algoritma.

In [17]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={"width_ratios": [2.2, 1]})
w0 = pd.Timestamp("2017-07-17"); sl = (te.index >= w0) & (te.index < w0 + pd.Timedelta(days=14))
ax[0].plot(te.index[sl], te.y[sl], label="Aktual", lw=1.6); ax[0].plot(te.index[sl], pred[sl], label="Prediksi HGB (day-ahead)", lw=1.2)
ax[0].plot(te.index[sl], te.lag24[sl], label="Baseline lag24", lw=.9, alpha=.6, ls="--"); ax[0].set_title("Dua pekan contoh (17-30 Jul 2017, musim panas)"); ax[0].set_ylabel("MW"); ax[0].legend(loc="upper left", fontsize=8)
ax[1].scatter(te.y, pred, s=2, alpha=.25); lo, hi = te.y.min(), te.y.max(); ax[1].plot([lo, hi], [lo, hi], "r--", lw=1); ax[1].set_xlabel("Aktual (MW)"); ax[1].set_ylabel("Prediksi (MW)"); ax[1].set_title("Aktual vs prediksi (test)")
plt.tight_layout(); plt.savefig("results/fig_forecast.png", bbox_inches="tight"); plt.show()


Pada gambar di atas (kiri), prediksi HGB mengikuti siklus harian beban aktual dengan baik, termasuk penurunan beban pada akhir pekan (22–23 Juli dan 29–30 Juli 2017). *Baseline* lag24 tampak tertinggal satu hari: pada peralihan dari hari kerja ke akhir pekan, *baseline* masih membawa level beban hari kerja sehingga prediksinya jauh di atas beban aktual. HGB dapat menyesuaikan karena memperoleh informasi hari dalam pekan. Meskipun demikian, HGB belum sempurna. Pada beberapa hari, misalnya sekitar 25 Juli dan malam hari 30 Juli, selisih antara prediksi dan beban aktual masih terlihat jelas.

Pada gambar di atas (kanan), titik-titik berkumpul di sepanjang garis diagonal, yang menunjukkan bahwa prediksi berkorelasi kuat dengan nilai aktual. Sebaran titik semakin lebar pada beban di atas sekitar 45.000 MW, yaitu kondisi beban puncak. Dari sisi operasi sistem, kondisi ini penting karena beban puncak merupakan saat cadangan daya paling dibutuhkan, tetapi pada saat itu pula ketidakpastian prediksi paling besar.

#### 3.2.3 Ablation Study

In [18]:
# ---- Ablation study: set feature mana yang layak? Dinilai dengan cross-validation kronologis DI DALAM TRAIN (independen dari test), lalu dilaporkan juga di test ----
abl = []
for nm, cols in SETS.items():
    cvm = []
    for a, b in TimeSeriesSplit(4).split(tr):
        m = HistGradientBoostingRegressor(random_state=SEED, **BP).fit(tr.iloc[a][cols], tr.iloc[a].y)
        cvm.append(100 * mean_absolute_error(tr.iloc[b].y, m.predict(tr.iloc[b][cols])) / tr.iloc[b].y.mean())
    m = HistGradientBoostingRegressor(random_state=SEED, **BP).fit(tr[cols], tr.y); mt = metrics(T, m.predict(te[cols]))
    abl.append(dict(set=nm, n_fitur=len(cols), cv_rel_pct=float(np.mean(cvm)), **mt))
abl = pd.DataFrame(abl); print(abl.round(3).to_string(index=False)); RES["ablation"] = abl.to_dict("records")


                           set  n_fitur  cv_rel_pct      MAE     RMSE    R2   rel
              A. Kalender saja        6       7.727 2995.296 3766.641 0.659 9.563
    B. Kalender + lag >=24 jam       12       4.893 1743.279 2385.486 0.863 5.566
C. B + fitur saat-terbit (D-1)       21       3.589 1274.602 1868.668 0.916 4.069


Pada tabel di atas, penggunaan *feature* kalender saja (set A) menghasilkan MAE 2.995 MW dan R² 0,659, yang lebih buruk daripada *baseline* lag24 (MAE 2.209 MW). Artinya, informasi kalender saja tidak cukup untuk memprediksi beban. Penambahan *lag* (set B) menurunkan MAE menjadi 1.743 MW, dan penambahan *snapshot* D-1 (set C) menurunkannya lagi menjadi 1.275 MW, atau 26,9% lebih rendah dibanding set B. Urutan yang sama muncul pada *cross-validation* di dalam data *training* (7,73%, 4,89%, dan 3,59%), sehingga peningkatan ini tidak hanya terjadi pada satu periode *testing*. Dari sudut pandang sistem tenaga, hasil ini menunjukkan bahwa kondisi beban paling akhir sebelum prediksi dibuat, yaitu pukul 23:00 hari D-1, memberikan informasi tambahan yang besar.

#### 3.2.4 Evaluasi Rolling-Origin

In [19]:
# ---- Evaluasi rolling-origin: 8 fold (train meluas, testing = 6 bulan berikutnya), semua data ----
rows = []
for i, (a, b) in enumerate(TimeSeriesSplit(8, test_size=4380).split(d)):
    A_, B_ = d.iloc[a], d.iloc[b]
    m = HistGradientBoostingRegressor(random_state=SEED, **BP).fit(A_[FINAL], A_.y); pm = m.predict(B_[FINAL]); mu = B_.y.mean()
    rows.append(dict(lipatan=i + 1, uji_mulai=str(B_.index[0].date()), uji_akhir=str(B_.index[-1].date()),
                     rel_HGB=100 * mean_absolute_error(B_.y, pm) / mu, rel_lag24=100 * mean_absolute_error(B_.y, B_.lag24) / mu, R2_HGB=r2_score(B_.y, pm)))
ro = pd.DataFrame(rows); print(ro.round(3).to_string(index=False))
print("\nrata-rata: HGB %.2f%% (std %.2f) | lag24 %.2f%% | R2 %.3f" % (ro.rel_HGB.mean(), ro.rel_HGB.std(), ro.rel_lag24.mean(), ro.R2_HGB.mean()))
RES["rolling"] = dict(rel_hgb_mean=float(ro.rel_HGB.mean()), rel_hgb_std=float(ro.rel_HGB.std()), rel_lag24_mean=float(ro.rel_lag24.mean()), r2_mean=float(ro.R2_HGB.mean()),
                      hgb_wins=int((ro.rel_HGB < ro.rel_lag24).sum()), n=int(len(ro)), rows=ro.round(3).to_dict("records"))


 lipatan  uji_mulai  uji_akhir  rel_HGB  rel_lag24  R2_HGB
       1 2014-08-04 2015-02-02    3.645      6.572   0.914
       2 2015-02-02 2015-08-03    4.222      7.615   0.912
       3 2015-08-04 2016-02-02    3.423      6.395   0.941
       4 2016-02-02 2016-08-02    3.895      6.654   0.929
       5 2016-08-03 2017-02-01    3.706      6.492   0.931
       6 2017-02-01 2017-08-02    4.332      7.503   0.909
       7 2017-08-03 2018-02-01    4.034      6.801   0.906
       8 2018-02-01 2018-08-02    4.763      7.990   0.876

rata-rata: HGB 4.00% (std 0.43) | lag24 7.00% | R2 0.915


Pada tabel di atas, HGB lebih baik daripada *baseline* lag24 pada 8 dari 8 *fold*. *Relative error* HGB berkisar antara 3,42% dan 4,76% dengan rata-rata 4,00% (simpangan baku 0,43), sedangkan *baseline* lag24 rata-rata 7,00%. R² rata-rata HGB adalah 0,915. Hasil ini menunjukkan bahwa keunggulan HGB konsisten sepanjang waktu dan tidak hanya berlaku pada satu periode *testing*. Rata-rata *rolling-origin* (4,00%) juga dekat dengan hasil pada *testing* utama (4,07%). *Fold* terakhir (Februari–Agustus 2018) memiliki *error* tertinggi (4,76%) dan R² terendah (0,876). Namun, *baseline* lag24 juga paling buruk pada *fold* ini (7,99%), sehingga yang sulit adalah periodenya, bukan modelnya. Rasio *error* HGB terhadap *baseline* pada *fold* ini (0,60) memang sedikit lebih tinggi daripada pada *fold* lain (0,54–0,59), jadi keunggulan HGB sedikit menyempit tetapi tetap ada. Pola serupa tampak pada semua *fold* genap (Februari–Agustus) dengan rata-rata *relative error* HGB 4,30% dibanding 3,70% pada *fold* ganjil (Agustus–Februari), dan *baseline* lag24 mengikuti pola yang sama (7,44% dibanding 6,57%). Penyebab pastinya tidak dilakukan riset lebih lanjut, tetapi pola ini menunjukkan bahwa kenaikan *error* pada *fold* 2018 merupakan variasi musiman yang juga muncul pada *fold* sebelumnya, bukan penurunan kualitas model.

#### 3.2.5 Analisis Error dan Feature Importance

In [20]:
# ---- Analisis error (data testing) ----
E = te[["y"]].copy(); E["pred"] = pred; E["err"] = E.pred - E.y; E["ae"] = E.err.abs(); E["pe"] = 100 * E.ae / E.y
E["hour"] = te.hour; E["dow"] = te.dow; E["month"] = te.month; E["is_hol"] = te.is_hol; E["date"] = te.index.date
by_hour = E.groupby("hour").pe.mean(); by_dow = E.groupby("dow").pe.mean(); by_month = E.groupby("month").pe.mean()
daytype = E.assign(tipe=np.where(E.is_hol == 1, "Libur federal", np.where(E.dow >= 5, "Akhir pekan", "Hari kerja"))).groupby("tipe").pe.mean()
daily_err = E.groupby("date").agg(mape=("pe", "mean"), mean_y=("y", "mean")); worst = daily_err.sort_values("mape", ascending=False).head(10)
print("MAPE per tipe hari:", daytype.round(2).to_dict()); print("MAPE per bulan:", by_month.round(2).to_dict())
print("10 hari terburuk:\n", worst.round(2).to_string()); print("bias rata-rata (pred - aktual): %.0f MW" % E.err.mean())
wk10 = int(sum(pd.Timestamp(i).dayofweek >= 5 for i in worst.index)); print("dari 10 hari terburuk, %d jatuh pada akhir pekan" % wk10)
print("P90 error absolut: %.0f MW | P99: %.0f MW | maks: %.0f MW" % (E.ae.quantile(.9), E.ae.quantile(.99), E.ae.max()))
RES["errors"] = dict(by_hour=[float(v) for v in by_hour.values], by_dow=[float(v) for v in by_dow.values], by_month=[float(v) for v in by_month.values],
                     daytype={k: float(v) for k, v in daytype.items()}, bias=float(E.err.mean()), p90=float(E.ae.quantile(.9)), p99=float(E.ae.quantile(.99)), max=float(E.ae.max()),
                     worst=[dict(date=str(i), mape=float(r.mape)) for i, r in worst.head(5).iterrows()], worst10_weekend=wk10)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].bar(by_hour.index, by_hour.values); ax[0].set_title("MAPE per jam (%)"); ax[0].set_xlabel("jam")
ax[1].bar(by_month.index, by_month.values); ax[1].set_title("MAPE per bulan (%)"); ax[1].set_xlabel("bulan")
ax[2].hist(E.err, bins=80); ax[2].set_title("Distribusi error (MW), prediksi - aktual"); ax[2].axvline(0, c="k", lw=.8)
plt.tight_layout(); plt.savefig("results/fig_errors.png", bbox_inches="tight"); plt.show()


MAPE per tipe hari: {'Akhir pekan': 4.3, 'Hari kerja': 3.68, 'Libur federal': 4.31}
MAPE per bulan: {1: 4.0, 2: 4.2, 3: 3.81, 4: 3.39, 5: 4.18, 6: 4.77, 7: 4.37, 8: 3.81, 9: 4.22, 10: 2.9, 11: 3.05, 12: 3.3}
10 hari terburuk:
              mape    mean_y
date                       
2017-05-20  26.40  26228.58
2018-06-03  17.88  26070.29
2016-09-24  17.45  27467.00
2015-05-20  17.24  28303.79
2018-05-05  16.76  26163.12
2018-05-27  15.25  28984.21
2015-02-15  14.22  40795.88
2017-07-29  13.86  29184.42
2015-05-13  13.27  27868.71
2016-06-12  12.81  33438.42
bias rata-rata (pred - aktual): -92 MW
dari 10 hari terburuk, 8 jatuh pada akhir pekan
P90 error absolut: 2900 MW | P99: 6681 MW | maks: 13556 MW


Pada gambar di atas (kiri), MAPE naik dari 1,75% pada pukul 00:00 menjadi 5,56% pada pukul 17:00, dan bertahan di sekitar 5,1% pada malam hari. Jam-jam awal memiliki *error* kecil karena dekat dengan *snapshot* pukul 23:00, sedangkan jam-jam berikutnya berjarak semakin jauh dari data terakhir yang diketahui, sampai 24 jam. Selain itu, sore hari merupakan saat beban puncak yang secara fisik lebih bervariasi. Pada panel tengah, MAPE tertinggi terjadi pada Juni (4,77%) dan terendah pada Oktober (2,90%). Bulan-bulan musim panas cenderung memiliki *error* lebih besar daripada Oktober sampai Desember. Hal ini konsisten dengan dugaan bahwa beban pendingin yang bergantung pada suhu sulit diprediksi tanpa data cuaca, tetapi dugaan ini tidak diuji.

MAPE menurut jenis hari menunjukkan bahwa hari kerja memiliki *error* terkecil (3,68%), sedangkan akhir pekan (4,30%) dan hari libur nasional (4,31%) lebih besar. Dari 10 hari dengan MAPE harian terburuk, 8 jatuh pada akhir pekan, dengan yang terburuk pada 2017-05-20 (MAPE 26,4%). Hari-hari tersebut berbeda dari pola hari kerja yang mendominasi data sehingga lebih sulit diprediksi.

Pada gambar di atas (kanan), distribusi *error* berpusat di sekitar nol dan hampir simetris, dengan bias rata-rata -92 MW (sekitar 0,3% dari beban rata-rata), artinya model cenderung sedikit lebih rendah dari beban aktual. Pada 90% jam, *error* absolut tidak melebihi sekitar 2.900 MW, dan pada 99% jam tidak melebihi sekitar 6.681 MW, sedangkan *error* terbesar mencapai 13.556 MW. Ekor distribusi yang panjang menunjukkan bahwa *error* besar jarang terjadi, tetapi tetap mungkin terjadi.

In [21]:
# ---- Permutation importance (sub-sampel 6.000 jam test, MAE) ----
rng = np.random.RandomState(SEED); sub = rng.choice(len(te), 6000, replace=False)
pi = permutation_importance(hgb, te[FINAL].iloc[sub], te.y.iloc[sub], n_repeats=5, random_state=SEED, scoring="neg_mean_absolute_error")
imp = pd.Series(pi.importances_mean, index=FINAL).sort_values(ascending=False); print(imp.round(1).head(12).to_string())
RES["importance"] = {k: float(v) for k, v in imp.head(10).items()}
fig, ax = plt.subplots(figsize=(7, 4)); imp.head(12)[::-1].plot.barh(ax=ax); ax.set_title("Permutation importance (kenaikan MAE, MW)"); plt.tight_layout(); plt.savefig("results/fig_importance.png", bbox_inches="tight"); plt.show()


last_obs       3359.4
lag24          2621.7
dow             428.7
d1_mean         410.9
hour            371.2
wk_mean         348.2
w1_mean         313.1
d1_min          266.0
dmean_prev      149.6
last_obs_m6     135.6
lag168          106.4
month           105.1


Pada gambar di atas, dua *feature* teratas adalah last_obs (MAE naik 3.359 MW saat diacak) dan lag24 (2.622 MW). Keduanya masing-masing adalah beban pukul 23:00 hari D-1 dan beban pada jam yang sama kemarin. Pengaruh kedua *feature* tersebut jauh lebih besar daripada dow (429 MW), d1_mean (411 MW), dan hour (371 MW). Hal ini sesuai dengan hasil *ablation study*: kondisi beban terbaru adalah informasi paling berharga, sedangkan kalender berperan sebagai koreksi, misalnya untuk membedakan hari kerja dan akhir pekan. Nilai *permutation importance* perlu dibaca dengan hati-hati karena *feature* yang saling berkorelasi, seperti lag24 dan wk_mean, dapat saling menggantikan sehingga pengaruh masing-masing tampak lebih kecil.

#### 3.2.6 Hasil pada Sepuluh Wilayah

In [22]:
# ---- Jalankan pipeline yang sama untuk tiap wilayah (tuning hanya di train; testing bersama 2016-01-01 s.d. 2018-08-02) ----
for code in REGIONS:
    r_ = run_region(code, CLEAN[code], export=(code != "PJME"))        # PJME di sini hanya untuk membandingkan protokol sama; model utamanya ada di Bab 3.1 dan 3.2
    RES2["regions"][code].update({k: v for k, v in r_.items() if not k.startswith("_")})
    if code != "PJME":
        json.dump(dict(web=r_["_web"], model=r_["_model"], fixture=r_["_fixture"]), open(f"results/regions_out/{code}.json", "w"), separators=(",", ":"))
    h = r_["models"]["HistGradientBoosting (tuned)"]
    print(f"{code:7s} test {r_['split']['n_test']//24:4d} hari | MAE {h['MAE']:8.1f} MW | rel {h['rel']:.2f}% | R2 {h['R2']:.3f} | vs baseline {r_['skill']['mae_reduction_pct']:.1f}% | "
          f"rolling menang {r_['rolling']['hgb_wins']}/{r_['rolling']['n']} | k={r_['cluster']['k']} | leakage:{'ada' if r_['leak_test']['changed_features'] else 'tidak ada'} | {r_['seconds']}s")


PJME    test  945 hari | MAE   1298.1 MW | rel 4.16% | R2 0.909 | vs baseline 41.4% | rolling menang 8/8 | k=2 | leakage:tidak ada | 216s


AEP     test  945 hari | MAE    500.2 MW | rel 3.38% | R2 0.922 | vs baseline 46.1% | rolling menang 8/8 | k=2 | leakage:tidak ada | 186s


COMED   test  945 hari | MAE    478.0 MW | rel 4.23% | R2 0.887 | vs baseline 42.1% | rolling menang 8/8 | k=2 | leakage:tidak ada | 110s


DAYTON  test  945 hari | MAE     81.5 MW | rel 4.04% | R2 0.906 | vs baseline 49.9% | rolling menang 8/8 | k=2 | leakage:tidak ada | 184s


DEOK    test  945 hari | MAE    145.9 MW | rel 4.68% | R2 0.880 | vs baseline 35.7% | rolling menang 8/8 | k=2 | leakage:tidak ada | 116s


DOM     test  945 hari | MAE    567.9 MW | rel 5.05% | R2 0.892 | vs baseline 35.7% | rolling menang 8/8 | k=3 | leakage:tidak ada | 155s


DUQ     test  945 hari | MAE     62.6 MW | rel 3.97% | R2 0.904 | vs baseline 37.7% | rolling menang 8/8 | k=2 | leakage:tidak ada | 160s


EKPC    test  945 hari | MAE    104.1 MW | rel 7.05% | R2 0.843 | vs baseline 25.9% | rolling menang 8/8 | k=2 | leakage:tidak ada | 66s


FE      test  945 hari | MAE    281.6 MW | rel 3.64% | R2 0.904 | vs baseline 46.8% | rolling menang 8/8 | k=2 | leakage:tidak ada | 101s


PJMW    test  945 hari | MAE    221.4 MW | rel 3.95% | R2 0.902 | vs baseline 41.4% | rolling menang 8/8 | k=2 | leakage:tidak ada | 209s


In [23]:
# ---- Perbandingan antarwilayah (protokol sama) ----
rows = []
for c in REGIONS:
    r = RES2["regions"][c]; h = r["models"]["HistGradientBoosting (tuned)"]; bb = r["baseline"][r["best_baseline"]]
    rows.append(dict(wilayah=c, rata2_MW=r["split"]["test_mean"], hari_uji=r["split"]["n_test"] // 24, MAE=h["MAE"], rel_HGB=h["rel"], rel_base=bb["rel"], R2=h["R2"], R2_base=bb["R2"],
                     turun_MAE_pct=r["skill"]["mae_reduction_pct"], rolling_menang=f"{r['rolling']['hgb_wins']}/{r['rolling']['n']}", rolling_rel=r["rolling"]["rel_hgb_mean"], k=r["cluster"]["k"], silhouette=r["cluster"]["silhouette"]))
cmp_ = pd.DataFrame(rows); print(cmp_.round(3).to_string(index=False)); RES2["compare"] = cmp_.round(4).to_dict("records")
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
o_ = cmp_.sort_values("rata2_MW", ascending=False); xx = np.arange(len(o_))
ax[0].bar(xx - .2, o_.rel_base, .4, label="baseline terbaik"); ax[0].bar(xx + .2, o_.rel_HGB, .4, label="HGB"); ax[0].set_xticks(xx); ax[0].set_xticklabels(o_.wilayah, rotation=30); ax[0].set_ylabel("relative error (%)"); ax[0].set_title("Relative error per wilayah (urut: beban rata-rata besar -> kecil)"); ax[0].legend()
ax[1].scatter(cmp_.rata2_MW, cmp_.rel_HGB); [ax[1].annotate(w, (m, e), fontsize=9, xytext=(3, 3), textcoords="offset points") for w, m, e in zip(cmp_.wilayah, cmp_.rata2_MW, cmp_.rel_HGB)]
ax[1].set_xscale("log"); ax[1].set_xlabel("beban rata-rata test (MW, skala log)"); ax[1].set_ylabel("relative error HGB (%)"); ax[1].set_title("Relative error vs ukuran wilayah")
plt.tight_layout(); plt.savefig("results/fig_regions.png", bbox_inches="tight"); plt.show()
cor_ = float(np.corrcoef(np.log(cmp_.rata2_MW), cmp_.rel_HGB)[0, 1]); print("korelasi log(beban rata-rata) vs relative error HGB: %.2f (n=%d wilayah; indikatif saja)" % (cor_, len(cmp_))); RES2["size_corr"] = cor_


wilayah  rata2_MW  hari_uji      MAE  rel_HGB  rel_base    R2  R2_base  turun_MAE_pct rolling_menang  rolling_rel  k  silhouette
   PJME 31173.248       945 1298.061    4.164     7.110 0.909    0.774         41.437            8/8        3.998  2       0.494
    AEP 14782.931       945  500.211    3.384     6.281 0.922    0.753         46.124            8/8        3.284  2       0.441
  COMED 11293.229       945  477.955    4.232     7.306 0.887    0.724         42.069            8/8        3.998  2       0.533
 DAYTON  2016.057       945   81.524    4.044     8.073 0.906    0.667         49.913            8/8        3.955  2       0.450
   DEOK  3117.016       945  145.906    4.681     7.284 0.880    0.747         35.740            8/8        4.500  2       0.486
    DOM 11238.419       945  567.949    5.054     7.855 0.892    0.764         35.660            8/8        4.888  3       0.486
    DUQ  1576.974       945   62.587    3.969     6.375 0.904    0.783         37.749            

korelasi log(beban rata-rata) vs relative error HGB: -0.42 (n=10 wilayah; indikatif saja)


Pada tabel di atas, HGB mengalahkan *baseline* terbaik di 10 dari 10 wilayah, dengan penurunan MAE antara 25,9% dan 49,9%. Pada evaluasi *rolling-origin*, HGB menang pada 80 dari 80 *fold* gabungan seluruh wilayah. *Relative error* HGB terendah terdapat pada AEP (3,38%) dan tertinggi pada EKPC (7,05%). Pada gambar di atas (kiri), batang HGB lebih rendah daripada batang *baseline* di setiap wilayah. Nilai EKPC paling tinggi untuk kedua batang (*baseline* 9,51%), yang menunjukkan bahwa deret beban EKPC memang lebih sulit diprediksi dan bukan hanya model yang kurang cocok. EKPC memiliki beban rata-rata terkecil dan data *training* terpendek (sekitar 2,6 tahun), tetapi kedua kemungkinan penyebab ini belum diuji.

Pada gambar di atas (kanan), tidak tampak hubungan yang jelas antara ukuran wilayah dan *relative error*. Korelasi antara logaritma beban rata-rata dan *relative error* HGB adalah r = -0,42, tetapi hanya didasarkan pada 10 titik sehingga hanya bersifat indikatif. DUQ dan DAYTON memiliki beban rata-rata yang kecil seperti EKPC, namun *relative error*-nya sekitar 4%. Uji *perturbation* tidak menemukan *feature* yang berubah pada seluruh wilayah sehingga tidak ada indikasi *data leakage*. Hasil ini menunjukkan bahwa pola yang ditemukan pada PJME tidak khas untuk satu wilayah, tetapi berlaku pada sembilan wilayah lain dengan pipeline yang sama.

#### 3.2.7 Hasil Clustering
*Clustering* merupakan analisis pendukung yang tidak dipakai sebagai masukan model *regression*. Hasilnya disajikan dalam dua tahap. Tahap pertama adalah hasil *tuning preprocessing* dan perbandingan algoritma, yang dimuat dari hasil grid penuh di `cluster_tuning/` (sel pertama di bawah). Tahap kedua adalah hasil konfigurasi yang akhirnya dipilih (sel kedua).


In [24]:
# ---- Hasil grid lengkap (dijalankan terpisah dengan cluster_tune.py, +-45 menit) dan perbandingan antar-wilayah ----
import glob, os
full = pd.DataFrame(json.load(open("cluster_tuning/cluster_tune_PJME.json")))                  # 265 konfigurasi: 10 pra-pemrosesan x 6 algoritma x k = 2..6
okf = full[(full.stab >= 0.8) & (full.min_frac >= 0.05)].sort_values("score", ascending=False)
print("Grid lengkap PJME:", len(full), "konfigurasi,", len(okf), "memenuhi syarat (stabilitas >= 0,8 dan cluster terkecil >= 5% hari)")
cols = ["prep", "alg", "k", "score", "sil_own", "db", "stab", "min_frac", "recon_R2"]
print("\n10 teratas menurut aturan seleksi awal:"); print(okf[cols].head(10).round(3).to_string(index=False))
print("\nKonfigurasi awal (K-Means + standardisasi per jam):"); print(full[(full.prep == "std_per_jam (awal)") & (full.alg == "kmeans")][cols].round(3).to_string(index=False))
pairs = json.load(open("cluster_tuning/cluster_pairs_PJME.json"))
print("\nPerbandingan berpasangan pada 10 subsampel 70% (k = 2), selisih skor terhadap konfigurasi awal:")
for o in pairs: print(f"  {o['prep']:22s} {o['alg']:12s} skor {o['mean']:.3f} | selisih {o['diff_vs_kmeans_std']:+.4f} +- {o['diff_sd']:.4f} | lebih baik pada {o['wins']}/10 subsampel")
# Perbandingan 5 konfigurasi kandidat pada 10 wilayah (tiap konfigurasi memakai k terbaik menurut skor, di antara yang memenuhi syarat)
CAND = [("std_per_jam (awal)", "kmeans"), ("raw_MW", "kmeans"), ("bentuk_z + level", "kmeans"), ("std_per_jam (awal)", "agglo_ward"), ("std_per_jam (awal)", "gmm_diag")]
reg_rows = []
for f in sorted(glob.glob("cluster_tuning/cluster_tune_*.json")):
    reg = os.path.basename(f).replace("cluster_tune_", "").replace(".json", "")
    for r in json.load(open(f)): r["wilayah"] = reg; reg_rows.append(r)
rg = pd.DataFrame(reg_rows); rg = rg[[(p, a) in CAND for p, a in zip(rg.prep, rg.alg)]]
best_rg = []
for (reg, pn, alg), g in rg.groupby(["wilayah", "prep", "alg"]):
    e = g[(g.stab >= 0.8) & (g.min_frac >= 0.05)]; best_rg.append((e if len(e) else g).sort_values("score", ascending=False).iloc[0])
best_rg = pd.DataFrame(best_rg)
agg = best_rg.groupby(["prep", "alg"])[["score", "sil_own", "db", "stab", "recon_R2", "min_frac"]].mean().reindex(CAND)
print("\nRata-rata pada", best_rg.wilayah.nunique(), "wilayah:"); print(agg.round(3).to_string())
k2 = rg[rg.k == 2].pivot_table(index="wilayah", columns=["prep", "alg"], values=["score", "recon_R2", "stab", "db"])
cmp_raw = {m: int((k2[m][("raw_MW", "kmeans")] > k2[m][("std_per_jam (awal)", "kmeans")]).sum()) for m in ["score", "recon_R2", "stab"]}
cmp_raw["db_lebih_kecil"] = int((k2["db"][("raw_MW", "kmeans")] < k2["db"][("std_per_jam (awal)", "kmeans")]).sum())
print("\nK-Means tanpa standardisasi vs standardisasi (k = 2, per wilayah, dari 10): ", cmp_raw)
RES["cluster_tuning"].update(
    pjme_top10=okf[cols].head(10).round(4).to_dict("records"), pjme_awal=full[(full.prep == "std_per_jam (awal)") & (full.alg == "kmeans")][cols].round(4).to_dict("records"),
    pjme_n_total=int(len(full)), pjme_n_eligible=int(len(okf)), pairs=pairs,
    best_per_alg=full[(full.stab >= 0.8) & (full.min_frac >= 0.05)].sort_values("score", ascending=False).groupby("alg").head(1)[cols].round(4).to_dict("records"),
    best_per_prep=full[(full.stab >= 0.8) & (full.min_frac >= 0.05)].sort_values("score", ascending=False).groupby("prep").head(1)[cols].round(4).to_dict("records"),
    regions=[dict(prep=p, alg=a, **{k: float(v) for k, v in agg.loc[(p, a)].items()}) for p, a in CAND], n_regions=int(best_rg.wilayah.nunique()),
    score_by_region={reg: {f"{p}|{a}": float(v) for (p, a), v in g.set_index(["prep", "alg"]).score.items()} for reg, g in best_rg.groupby("wilayah")},
    raw_vs_std=cmp_raw)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
lab_cfg = {"std_per_jam (awal)|kmeans": "K-Means\nstandar (awal)", "raw_MW|kmeans": "K-Means\nMW asli", "bentuk_z + level|kmeans": "K-Means\nshape + level", "std_per_jam (awal)|agglo_ward": "Agglomerative\nWard", "std_per_jam (awal)|gmm_diag": "GMM\ndiagonal"}
x = np.arange(len(CAND)); w = .26
for i, (m, nm) in enumerate([("score", "Silhouette (ruang netral)"), ("sil_own", "Silhouette (ruang sendiri)"), ("recon_R2", "R² rekonstruksi profil")]):
    ax[0].bar(x + (i - 1) * w, [agg.loc[c, m] for c in CAND], w, label=nm)
ax[0].set_xticks(x); ax[0].set_xticklabels([lab_cfg[f"{p}|{a}"] for p, a in CAND], fontsize=8); ax[0].set_title(f"Rata-rata {best_rg.wilayah.nunique()} wilayah (k terbaik tiap konfigurasi)"); ax[0].legend(fontsize=8)
ax[1].scatter(full.score, full.recon_R2, s=14, alpha=.35, label="semua konfigurasi PJME")
aw = full[(full.prep == "std_per_jam (awal)") & (full.alg == "kmeans")]; rw = full[(full.prep == "raw_MW") & (full.alg == "kmeans")]
ax[1].scatter(aw.score, aw.recon_R2, s=45, c="tab:orange", label="K-Means standar (awal)"); ax[1].scatter(rw.score, rw.recon_R2, s=45, c="tab:red", marker="s", label="K-Means MW asli (terpilih)")
ax[1].set_xlabel("skor Silhouette ruang netral"); ax[1].set_ylabel("R² rekonstruksi profil (MW)"); ax[1].set_title("PJME: kualitas pemisahan vs kemampuan mewakili profil"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("results/fig_cluster_tuning.png", bbox_inches="tight"); plt.show()


Grid lengkap PJME: 265 konfigurasi, 138 memenuhi syarat (stabilitas >= 0,8 dan cluster terkecil >= 5% hari)

10 teratas menurut aturan seleksi awal:
              prep      alg  k  score  sil_own    db  stab  min_frac  recon_R2
std_per_jam (awal) spectral  2  0.282    0.472 0.821 0.844     0.327     0.568
  bentuk_z + level   kmeans  2  0.279    0.335 1.183 0.979     0.332     0.552
            raw_MW   kmeans  2  0.276    0.494 0.780 0.988     0.344     0.573
    minmax_per_jam   kmeans  2  0.273    0.480 0.807 0.989     0.355     0.570
    robust_per_jam   kmeans  2  0.273    0.480 0.808 0.990     0.352     0.568
  bentuk_z + level gmm_diag  2  0.273    0.277 1.369 0.965     0.326     0.273
 bentuk_z_per_hari gmm_diag  2  0.271    0.307 1.064 0.943     0.336     0.254
std_per_jam (awal)   kmeans  2  0.271    0.477 0.809 0.972     0.363     0.562
    pca_3_komponen   kmeans  2  0.271    0.489 0.784 0.977     0.364     0.561
       pca_95%_var   kmeans  2  0.271    0.489 0.784 0.977   


Perbandingan berpasangan pada 10 subsampel 70% (k = 2), selisih skor terhadap konfigurasi awal:
  std_per_jam (awal)     kmeans       skor 0.271 | selisih +0.0000 +- 0.0000 | lebih baik pada 0/10 subsampel
  std_per_jam (awal)     spectral     skor 0.279 | selisih +0.0073 +- 0.0031 | lebih baik pada 10/10 subsampel
  bentuk_z + level       kmeans       skor 0.279 | selisih +0.0073 +- 0.0023 | lebih baik pada 10/10 subsampel
  bentuk_z_per_hari      gmm_diag     skor 0.271 | selisih +0.0002 +- 0.0034 | lebih baik pada 5/10 subsampel
  std_per_jam (awal)     agglo_ward   skor 0.272 | selisih +0.0002 +- 0.0132 | lebih baik pada 8/10 subsampel
  std_per_jam (awal)     gmm_diag     skor 0.256 | selisih -0.0148 +- 0.0021 | lebih baik pada 0/10 subsampel
  raw_MW                 kmeans       skor 0.276 | selisih +0.0050 +- 0.0008 | lebih baik pada 10/10 subsampel

Rata-rata pada 10 wilayah:
                               score  sil_own     db   stab  recon_R2  min_frac
prep               alg

**Perbandingan algoritma.** Pada tabel di atas, *spectral clustering* memperoleh *score* tertinggi (0,282), diikuti K-Means pada bentuk + level (0,279) dan *GMM* diagonal (0,273). Selisih tiga kandidat teratas hanya 0,010, sedangkan *GMM full* (0,242) dan Ward (0,232) lebih rendah. Semua algoritma memperoleh *score* tertinggi pada k = 2, dan *score* turun ketika k bertambah. Ward hanya memenuhi syarat stabilitas pada k = 2. *Agglomerative* dengan *linkage average* tidak punya konfigurasi yang memenuhi syarat: *score* tertinggi di seluruh grid (0,456) berasal dari konfigurasi ini, tetapi *cluster* terkecilnya hanya berisi sekitar 3 hari dan stabilitasnya 0,15. Artinya algoritma ini hanya memisahkan beberapa hari ekstrem dari sisanya, yang tidak berguna sebagai segmentasi. Secara keseluruhan 127 dari 265 konfigurasi gugur karena syarat tersebut.

**Pengaruh *preprocessing*.** Tabel di atas menunjukkan bahwa *preprocessing* hampir tidak mengubah hasil K-Means. Pada k = 2, tujuh varian yang mempertahankan level beban (MW asli, standardisasi, *robust*, *min-max*, log, dan dua varian PCA) menghasilkan *score* 0,260–0,276 dan R² rekonstruksi 0,55–0,57. Simpangan baku beban per jam berkisar 3.665–6.984 MW sehingga standardisasi memang mengubah bobot tiap jam, tetapi pembagian harian hampir tidak berubah. Dugaan kami, variasi dominan antarhari (hari tinggi atau rendah) bergerak searah pada semua jam sehingga bobot per jam tidak banyak mengubah urutan kedekatan antarhari; dugaan ini tidak diuji lebih lanjut. Varian bentuk saja berbeda: R² rekonstruksinya hanya 0,10 (z-score per hari) dan 0,25 (rasio) karena level beban dibuang, sehingga rata-rata *cluster* tidak lagi mewakili profil dalam MW. Hasil *cluster*-nya berbeda secara kualitatif: pengelompokan menurut bentuk lebih berkaitan dengan bulan (NMI terhadap bulan 0,25 dibanding 0,13 pada MW asli), tetapi tidak memisahkan hari berdasarkan tinggi beban. Varian bentuk + level mengembalikan R² ke 0,55.

**Uji perbedaan antar-kandidat.** Perbedaan antarkandidat diuji pada 10 *subsample* 70% (k = 2), dengan selisih *score* terhadap K-Means standar dihitung pada *subsample* yang sama. Pada tabel di atas, *spectral clustering* (+0,007), bentuk + level (+0,007), dan K-Means MW asli (+0,005) mengungguli K-Means standar pada 10, 10, dan 10 dari 10 *subsample*. Konsisten, tetapi selisihnya hanya sekitar 0,005–0,007 pada skala Silhouette yang berkisar antara −1 dan 1, sehingga secara praktis tidak mengubah pembagian hari. Ward tidak berbeda dari K-Means standar (+0,000 ± 0,013, simpangan besar), sedangkan *GMM* diagonal pada data standar lebih rendah (−0,015, tidak pernah lebih baik).

**Konsistensi pada sepuluh wilayah.** Pada tabel di atas, bentuk + level memiliki *score* tertinggi (0,304) tetapi R² rekonstruksi paling rendah (0,41 dibanding 0,55 pada konfigurasi awal). *Score*-nya juga cenderung menguntungkan konfigurasi ini karena salah satu ruang pembanding (S2) adalah ruang bentuk, yaitu ruang yang dipakai konfigurasi tersebut. K-Means MW asli lebih tinggi daripada K-Means standar (0,277 vs 0,267), dan R² rekonstruksinya juga lebih tinggi (0,56 vs 0,55). Pada k = 2 per wilayah, MW asli unggul dalam *score* di 7 dari 10 wilayah dan dalam R² rekonstruksi di 10 dari 10 wilayah. Ward (0,254) dan *GMM* diagonal (0,246) lebih rendah, dan Ward tidak stabil (stabilitas rata-rata 0,52).

Pada gambar di atas (kiri), K-Means MW asli lebih tinggi daripada K-Means standar pada ketiga ukuran, sedangkan bentuk + level unggul pada *score* tetapi turun jauh pada Silhouette ruang sendiri dan R² rekonstruksi. Pada gambar di atas (kanan), dari 24 konfigurasi dengan *score* ≥ 0,25 dan R² ≥ 0,5, sebanyak 23 berada pada k = 2, dan semuanya menumpuk pada *score* 0,25–0,28; tidak ada konfigurasi yang jauh lebih baik pada kedua sumbu sekaligus. Untuk k yang lebih besar, R² rekonstruksi naik (karena lebih banyak *cluster* berarti rata-rata yang lebih rinci) tetapi *score* turun. Titik-titik di dasar grafik dengan R² mendekati nol adalah konfigurasi yang memisahkan segelintir hari saja (termasuk *agglomerative average* di atas), sehingga tidak berguna meskipun *score*-nya tinggi.

**Konfigurasi yang dipilih.** Aturan seleksi yang ditetapkan sebelumnya akan memilih *spectral clustering* pada data standar (k = 2, *score* 0,282). Konfigurasi ini tidak dipilih. Alasannya: selisihnya terhadap K-Means hanya +0,007, stabilitasnya paling rendah di antara kandidat teratas (0,84 dibanding 0,99 pada K-Means MW asli), dan implementasi *spectral clustering* di scikit-learn tidak menyediakan fungsi *predict* untuk hari baru. Bentuk + level tidak dipilih karena R² rekonstruksinya jauh lebih rendah pada rata-rata sepuluh wilayah. Konfigurasi yang dipilih adalah K-Means pada profil dalam MW tanpa standardisasi (PJME: *score* 0,276, stabilitas 0,99, R² rekonstruksi 0,57), dengan alasan: *score* dan R² rekonstruksi lebih baik daripada konfigurasi awal secara konsisten, stabilitasnya tinggi (setara varian K-Means lain, 0,97–0,99), dapat dipakai pada hari baru, dan lebih sederhana. Ini adalah keputusan pertimbangan yang diambil setelah melihat hasil, bukan keluaran otomatis dari aturan seleksi, sehingga perlu dicatat sebagai keputusan tim.

Hasil tuning menunjukkan tidak ada model yang jelas lebih baik daripada konfigurasi awal. Perubahan yang dipilih berupa perbaikan kecil yang konsisten, bukan lompatan kualitas. Kedua partisi juga mirip (ARI 0,82 pada k = 2 di PJME).


In [25]:
# ---- Konfigurasi terpilih: K-Means pada profil dalam MW (tanpa standardisasi); k dari Silhouette tertinggi ----
srows, K_BEST, lab = cluster_final(M)
sil = pd.DataFrame(srows); print(sil.round(3).to_string(index=False))
RES["silhouette"] = sil.round(4).to_dict("records")
names = {0: "Beban rendah", 1: "Beban tinggi"} if K_BEST == 2 else {i: f"Cluster {i}" for i in range(K_BEST)}
info = pd.DataFrame({"date": dates, "cl": lab}); info["dow"] = info.date.dt.dayofweek; info["month"] = info.date.dt.month
info["weekend"] = (info.dow >= 5).astype(int); info["mean_mw"] = M.mean(1); info["peak_h"] = M.argmax(1)
hol = holidays.US(years=range(2001, 2020)); info["holiday"] = info.date.dt.date.map(lambda x: int(x in hol))
summ = info.groupby("cl").agg(n_hari=("cl", "size"), rata2_MW=("mean_mw", "mean"), persen_akhir_pekan=("weekend", "mean"),
                              persen_libur=("holiday", "mean"), jam_puncak_median=("peak_h", "median")).round(3)
print(summ.to_string())
print("\nKomposisi bulan (% hari dalam cluster):"); print((pd.crosstab(info.cl, info.month, normalize="index") * 100).round(0).astype(int).to_string())
sk = sil.loc[sil.k == K_BEST].iloc[0]
print("Silhouette k=%d: %.3f (konfigurasi awal: %.3f) | kesesuaian dengan konfigurasi awal (ARI): %.3f" % (K_BEST, sk.silhouette, sk.silhouette_std, sk.ari_vs_std))
RES["cluster"] = dict(k=K_BEST, silhouette=float(sk.silhouette), silhouette_std=float(sk.silhouette_std), ari_vs_std=float(sk.ari_vs_std), summary=summ.reset_index().to_dict("records"),
                      month_pct=(pd.crosstab(info.cl, info.month, normalize="index") * 100).round(1).to_dict("index"))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for c in range(K_BEST):
    prof = M[lab == c]; ax[0].plot(prof.mean(0), lw=2, label=f"{names[c]} (n={len(prof)})"); ax[0].fill_between(range(24), np.percentile(prof, 10, 0), np.percentile(prof, 90, 0), alpha=.15)
ax[0].set_title("Profil beban tiap cluster (garis = rata-rata, area = P10-P90)"); ax[0].set_xlabel("jam"); ax[0].set_ylabel("MW"); ax[0].legend()
ax[1].plot(sil.k, sil.silhouette, "o-", label="K-Means MW asli (terpilih)"); ax[1].plot(sil.k, sil.silhouette_std, "s--", label="K-Means standar (awal)"); ax[1].set_title("Silhouette vs k"); ax[1].set_xlabel("k"); ax[1].legend()
pc = PCA(2).fit_transform(M); ax[2].scatter(pc[:, 0], pc[:, 1], c=lab, s=4, cmap="coolwarm"); ax[2].set_title("Proyeksi PCA 2D (warna = cluster)")
plt.tight_layout(); plt.savefig("results/fig_cluster.png", bbox_inches="tight"); plt.show()


 k  silhouette  davies_bouldin  min_cluster  silhouette_std  ari_vs_std
 2       0.494           0.780         2083           0.477       0.824
 3       0.412           0.849          904           0.383       0.781
 4       0.324           1.001          701           0.314       0.712
 5       0.344           0.949          483           0.327       0.766
 6       0.321           0.968          444           0.317       0.750
    n_hari   rata2_MW  persen_akhir_pekan  persen_libur  jam_puncak_median
cl                                                                        
0     3974  29316.132               0.348         0.035               19.0
1     2083  37350.260               0.167         0.021               17.0

Komposisi bulan (% hari dalam cluster):
month  1   2   3   4   5   6   7   8   9   10  11  12
cl                                                   
0       6   7  12  13  12   6   2   3   8  12  12   8
1      14  10   3   0   3  13  22  19   7   1   1   8
Silhouette 

Pada tabel di atas, Silhouette Score konfigurasi terpilih tertinggi pada k = 2 (0,494) dan menurun pada k yang lebih besar (0,412 pada k = 3 dan 0,325 pada k = 4). Davies-Bouldin Index, yang bernilai lebih baik jika lebih kecil, juga terendah pada k = 2 (0,780). Kedua ukuran sama-sama memilih k = 2. Silhouette konfigurasi terpilih lebih tinggi daripada konfigurasi awal pada semua k, tetapi keduanya dihitung di ruang feature masing-masing, sehingga selisih ini tidak dapat dibaca sebagai peningkatan kualitas sebesar itu; pembanding yang adil adalah *score* di ruang netral pada hasil uji berpasangan di atas. Nilai Silhouette 0,494 juga masih jauh dari 1 sehingga pemisahan antar-*cluster* tidak tegas.

Pada tabel di atas, *cluster* beban rendah berisi 3.974 hari dengan rata-rata 29.316 MW, dan 34,8% di antaranya jatuh pada akhir pekan. *Cluster* beban tinggi berisi 2.083 hari dengan rata-rata 37.350 MW, dan hanya 16,7% di antaranya jatuh pada akhir pekan. Jam puncak median *cluster* beban tinggi adalah pukul 17:00, lebih awal dibanding *cluster* beban rendah (pukul 19:00). Pada gambar di atas (kiri), profil *cluster* beban tinggi berada di atas profil *cluster* beban rendah sepanjang hari dan mencapai sekitar 43.800 MW pada pukul 18:00. Area P10–P90 *cluster* beban tinggi juga lebih lebar (rata-rata sekitar 10.500 MW dibanding 7.700 MW), yang berarti variasi beban antarhari di dalam *cluster* ini lebih besar.

Dari sebaran bulan, 85,9% hari pada *cluster* beban tinggi berada pada Desember–Februari dan Juni–Agustus, sedangkan 68,7% hari pada *cluster* beban rendah berada pada Maret–Mei dan September–November. Dengan demikian, *cluster* beban tinggi dapat diartikan sebagai hari-hari ekstrem musim panas dan musim dingin, sedangkan *cluster* beban rendah berisi musim peralihan dan akhir pekan. Pada gambar di atas (tengah), penurunan Silhouette Score dari k = 2 ke k = 4 mengonfirmasi bahwa pembagian yang lebih halus tidak menghasilkan kelompok yang lebih jelas. Pada gambar di atas (kanan), titik-titik pada proyeksi PCA membentuk satu sebaran yang kontinu dan kedua *cluster* hanya terbelah di sekitar satu nilai pada sumbu PC1 tanpa celah di antaranya. Artinya, K-Means membagi satu kontinum hari menjadi dua bagian dan tidak menemukan dua kelompok yang terpisah secara alami.

In [26]:
# ---- Ringkasan clustering antarwilayah (konfigurasi terpilih) ----
rows_ = [dict(wilayah=c, k=RES2["regions"][c]["cluster"]["k"], silhouette=RES2["regions"][c]["cluster"]["silhouette"], silhouette_std=RES2["regions"][c]["cluster"]["silhouette_std"],
              ari_vs_std=RES2["regions"][c]["cluster"]["ari_vs_std"]) for c in REGIONS]
cl_tab = pd.DataFrame(rows_); print(cl_tab.round(3).to_string(index=False)); RES2["cluster_summary"] = cl_tab.round(4).to_dict("records")


wilayah  k  silhouette  silhouette_std  ari_vs_std
   PJME  2       0.494           0.477       0.824
    AEP  2       0.441           0.447       0.947
  COMED  2       0.533           0.485       0.716
 DAYTON  2       0.450           0.446       0.868
   DEOK  2       0.486           0.469       0.790
    DOM  3       0.486           0.479       0.939
    DUQ  2       0.493           0.467       0.761
   EKPC  2       0.526           0.478       0.752
     FE  2       0.467           0.452       0.812
   PJMW  2       0.448           0.454       0.962


Pada tabel di atas, Silhouette konfigurasi terpilih lebih tinggi daripada konfigurasi awal pada 8 dari 10 wilayah, dan kedua partisi sangat mirip (ARI 0,72–0,96). Perlu diingat bahwa kedua Silhouette dihitung di ruang feature masing-masing sehingga selisihnya tidak dapat dibaca sebagai peningkatan kualitas.

### 3.3 Evaluation
#### 3.3.1 Metrik Evaluasi
Evaluasi *regression* menggunakan empat ukuran berikut, dengan y adalah beban aktual, ŷ adalah beban prediksi, ȳ adalah rata-rata beban aktual, dan n adalah jumlah data.

$$MAE = \frac{1}{n} \sum |y - \hat{y}|$$
$$RMSE = \sqrt{\frac{1}{n} \sum (y - \hat{y})^2}$$
$$R^2 = 1 - \frac{\sum (y - \hat{y})^2}{\sum (y - \bar{y})^2}$$
$$Relative\ error = \frac{MAE}{\bar{y}} \times 100\%$$

MAE adalah rata-rata selisih absolut antara prediksi dan beban aktual dalam satuan MW. RMSE memberi bobot lebih besar pada *error* yang besar sehingga lebih sensitif terhadap kegagalan prediksi saat beban puncak. R² menunjukkan proporsi variasi beban yang dijelaskan model. *Relative error* menyatakan MAE sebagai persentase beban rata-rata sehingga dapat dibandingkan antarwilayah. MAPE (rata-rata dari |y − ŷ| / y) digunakan hanya pada analisis *error* menurut jam, bulan, dan jenis hari. Untuk *clustering* digunakan Silhouette Score, yaitu s = (b − a) / max(a, b) dengan a jarak rata-rata ke anggota *cluster* sendiri dan b jarak rata-rata ke *cluster* terdekat lainnya, serta Davies-Bouldin Index yang makin baik jika makin kecil.

#### 3.3.2 Arti Nilai Evaluasi
*Ringkasan evaluasi HGB dibanding baseline terbaik pada PJME.*

| Ukuran | *Baseline* terbaik (lag24) | HistGradientBoosting |
|---|---:|---:|
| MAE (MW) | 2.209 | **1.275** |
| RMSE (MW) | 3.025 | **1.869** |
| R² | 0,780 | **0,916** |
| *Relative error* (rata-rata beban *testing* 31.323 MW) | 7,05% | **4,07%** |

Berdasarkan tabel di atas, HGB menurunkan MAE sebesar 42,3% dan RMSE sebesar 38,2% dibanding *baseline* lag24. Penurunan ini merupakan ukuran kemampuan prediksi yang paling berarti. Nilai R² 0,916 terlihat tinggi, tetapi *baseline* lag24 saja sudah memperoleh R² 0,780 karena siklus harian beban mudah ditebak. Dengan demikian, R² tidak dapat dipakai sendirian untuk menilai keunggulan model. Hasil ini juga konsisten antarwaktu (HGB lebih baik pada 8 dari 8 *fold rolling-origin*) dan antarwilayah (10 dari 10 wilayah). Untuk *clustering*, Silhouette Score 0,494 dan analisis PCA menunjukkan bahwa pemisahan *cluster* tidak tegas, sehingga *cluster* berguna sebagai ringkasan deskriptif, bukan sebagai kelompok alami.

#### 3.3.3 Catatan terhadap Validitas Hasil
1. Set *feature* snapshot D-1 dirancang setelah percobaan awal yang juga melihat data *testing* PJME. Karena itu, angka *testing* PJME kemungkinan sedikit lebih optimistis. Ukuran yang lebih aman adalah rata-rata *rolling-origin* (4,00%) dan urutan peningkatan yang sama pada *cross-validation* di dalam data *training*.
2. Setiap wilayah hanya diuji pada satu periode sekitar 2,6 tahun tanpa data cuaca. Signifikansi statistik antarwilayah tidak diuji, dan wilayah yang berdekatan berbagi kondisi cuaca sehingga hasilnya tidak sepenuhnya independen.
3. Set *feature* dan rentang *hyperparameter* dirancang pada PJME dan dipakai apa adanya pada wilayah lain.
4. Nilai yang tidak masuk akal dibuang berdasarkan aturan, tidak berdasarkan verifikasi ke sumber asli.
5. Angka PJME memiliki dua protokol *testing*, yaitu 4,07% pada *testing* mulai 2015 dan 4,16% pada *testing* mulai 2016 untuk perbandingan antarwilayah.

#### 3.3.4 Evaluasi Clustering
Data tidak memiliki label jenis hari yang sebenarnya, sehingga *clustering* dievaluasi dengan ukuran internal. Silhouette Score s = (b − a) / max(a, b), dengan a adalah jarak rata-rata suatu hari ke hari lain dalam *cluster* yang sama dan b adalah jarak rata-rata ke *cluster* terdekat lainnya; nilai 1 berarti terpisah sempurna dan nilai mendekati 0 berarti *cluster* saling tumpang tindih. Davies-Bouldin Index menilai rasio sebaran dalam *cluster* terhadap jarak antar-*cluster* (lebih kecil lebih baik). Stabilitas diukur dengan *Adjusted Rand Index* (ARI) antar-dua-hasil *clustering* pada *subsample* berbeda (1 berarti pembagian identik). R² rekonstruksi = 1 − SSE / SST, yaitu proporsi variasi profil MW yang dijelaskan oleh rata-rata *cluster*.

Pada konfigurasi terpilih, Silhouette Score 0,494 (k = 2) menunjukkan pemisahan yang tidak tegas, stabilitas 0,99 menunjukkan bahwa pembagian hampir sama pada *subsample* berbeda, dan R² rekonstruksi 0,57 berarti rata-rata dua *cluster* menjelaskan sekitar 57% variasi profil harian. Bersama proyeksi PCA yang tidak menunjukkan celah antar-*cluster* (Subbab 3.2.7), angka-angka ini mendukung pembacaan hasil sebagai ringkasan deskriptif, bukan kelompok alami.

1. Silhouette Score tidak dapat dibandingkan langsung antar-*preprocessing* karena dihitung di ruang feature yang berbeda; perbandingan memakai *score* di dua ruang netral (Subbab 3.1.3).
2. Konfigurasi akhir dipilih dengan pertimbangan setelah hasil dilihat, bukan keluaran otomatis dari aturan seleksi yang ditetapkan sebelumnya (Subbab 3.2.7).
3. Tuning dan evaluasi memakai data yang sama karena analisis ini deskriptif dan tidak ada data *hold-out*, sehingga *score* konfigurasi pemenang cenderung sedikit optimistis. Uji berpasangan pada *subsample* dan pengulangan pada 9 wilayah mengurangi, tetapi tidak menghilangkan, risiko ini.
4. Jumlah *cluster* terbaik adalah k = 2 pada 9 dari 10 wilayah, sedangkan DOM memilih k = 3; tuning dilakukan paling lengkap pada PJME.


### 3.4 Engineering Interpretation
#### 3.4.1 Interpretasi Hasil dari Perspektif Teknik Elektro
Model HGB memprediksi beban per jam satu hari ke depan dengan MAE 1.275 MW atau 4,07% dari beban rata-rata PJME. Bagi operator sistem, angka ini berarti bahwa pada kondisi *testing*, jadwal pembangkit yang disusun dari prediksi tersebut rata-rata hanya meleset sekitar 4,1% dari kebutuhan beban sebenarnya, jauh lebih kecil dibanding 7,1% jika hanya mengandalkan beban kemarin. Pada 90% jam, selisih prediksi dan beban aktual tidak melebihi sekitar 2.900 MW, dan pada 99% jam tidak melebihi sekitar 6.681 MW. Besaran ini dapat menjadi gambaran awal ketidakpastian yang perlu diantisipasi melalui cadangan daya. Biaya atau besar cadangan yang sebenarnya tidak dihitung karena data tarif dan kebijakan cadangan tidak tersedia.

Pola pada hasil juga sejalan dengan perilaku sistem tenaga. Beban naik pada jam kerja dan turun pada malam hari serta akhir pekan (Subbab 3.2.1). *Error* terbesar muncul pada sore hari saat beban puncak (Subbab 3.2.5), dan hari-hari dengan *error* terburuk didominasi akhir pekan. Pada *clustering*, hari-hari beban tinggi terkonsentrasi pada musim panas dan musim dingin (Subbab 3.2.7), yang sesuai dengan dugaan bahwa beban pendingin dan pemanas ruangan membentuk beban puncak tahunan. Dugaan terkait suhu ini konsisten dengan data, tetapi tidak dapat dipastikan karena suhu tidak termasuk dalam *dataset*.

Informasi yang paling berharga bagi model adalah kondisi beban terbaru, yaitu beban pukul 23:00 hari sebelumnya dan beban pada jam yang sama kemarin (Subbab 3.2.3 dan 3.2.5). Secara praktis, hal ini berarti bahwa kualitas prediksi *day-ahead* sangat bergantung pada ketersediaan data pengukuran beban yang cepat dan akurat dari sistem SCADA sampai saat prediksi dibuat. Selain itu, perbedaan akurasi antara HGB, Random Forest, dan Ridge yang kecil menunjukkan bahwa model sederhana yang mudah dijelaskan sudah cukup memberikan hasil yang mendekati model yang lebih kompleks pada data ini.

#### 3.4.2 Manfaat Hasil AI
1. Prediksi beban per jam untuk 24 jam ke depan yang lebih akurat dibanding *baseline* sederhana, sehingga dapat mendukung penyusunan jadwal pembangkit.
2. Gambaran besar ketidakpastian prediksi (P90 dan P99 *error*) sebagai dasar awal diskusi mengenai cadangan daya.
3. Pengelompokan hari menjadi profil beban rendah dan tinggi yang membantu perencana mengenali hari-hari yang perlu mendapat perhatian lebih.

#### 3.4.3 Penggunaan dalam Sistem Teknik Elektro
Model dapat dijalankan setiap hari pada pukul 00:00 sebagai bagian dari perangkat bantu perencanaan operasi, misalnya untuk menyusun jadwal pembangkit (*unit commitment*) dan menentukan margin cadangan. Hasil *clustering* dapat dipakai untuk menandai hari yang diperkirakan masuk ke profil beban tinggi. Karena model dilatih terpisah untuk tiap wilayah dan pipeline-nya sama, pendekatan yang sama dapat diterapkan pada wilayah lain selama tersedia data beban per jam yang cukup panjang.

#### 3.4.4 Keterbatasan
1. Model tidak memakai data cuaca, padahal beban sangat dipengaruhi suhu. Prediksi pada hari dengan perubahan cuaca mendadak kemungkinan lebih buruk.
2. Data berasal dari satu jaringan di Amerika Serikat dengan iklim, kalender libur, dan kebiasaan konsumsi yang berbeda dari Indonesia (misalnya tidak ada musim dingin dan ada libur Lebaran). Hasil tidak dapat langsung diterapkan di Indonesia.
3. Setiap wilayah hanya diuji pada satu periode sekitar 2,6 tahun, dan beberapa keputusan perancangan dibuat pada PJME.
4. Model hanya memberikan satu nilai prediksi tanpa interval ketidakpastian, sedangkan *error* besar tetap dapat terjadi (maksimum 13.556 MW pada PJME).
5. Data tidak diverifikasi ke sumber asli PJM, dan beberapa nilai yang diragukan tetap dipertahankan.
6. Hasil *clustering* membagi satu kontinum hari menjadi dua bagian dan tidak ditemukan model *clustering* yang jelas lebih baik dari K-Means; perbaikan dari tuning bersifat kecil.

#### 3.4.5 Pengembangan untuk Kondisi Nyata
1. Menambahkan prakiraan suhu dan variabel cuaca lain sebagai *feature*.
2. Menghasilkan interval prediksi (*probabilistic forecasting*) agar penentuan cadangan daya dapat dikaitkan langsung dengan tingkat risiko.
3. Memvalidasi model dengan data beban Indonesia, dengan memasukkan kalender libur nasional dan hari besar keagamaan.
4. Menambahkan pemantauan akurasi secara berkelanjutan dan pelatihan ulang model secara berkala setelah model dipakai pada operasi nyata.
5. Menguji pada periode yang lebih panjang dan menguji signifikansi statistik antarwilayah.


## BAB 4. KESIMPULAN
Berdasarkan hasil pengujian, model HistGradientBoosting dengan *feature* kalender dan riwayat beban yang bebas dari *data leakage* mampu memprediksi beban PJME per jam satu hari ke depan dengan MAE 1.275 MW (*relative error* 4,07%), RMSE 1.869 MW, dan R² 0,916 pada periode 2015–2018. Nilai MAE tersebut 42,3% lebih rendah daripada *baseline* persistensi harian. Hasil ini konsisten pada 8 dari 8 *fold rolling-origin* dengan rata-rata *relative error* 4,00%, yang merupakan ukuran paling aman karena tidak bergantung pada satu periode *testing*.

*Ablation study* dan *permutation importance* menunjukkan bahwa peningkatan akurasi terutama berasal dari *feature* kondisi beban terbaru (beban pukul 23:00 hari sebelumnya dan beban pada jam yang sama kemarin), bukan dari kompleksitas algoritma, karena Random Forest dan Ridge menghasilkan R² yang berdekatan. Pipeline yang sama pada 10 wilayah menghasilkan *relative error* 3,38% sampai 7,05% dan HGB mengalahkan *baseline* terbaik di 10 dari 10 wilayah, sehingga hasil PJME tidak khas untuk satu wilayah.

*Clustering* K-Means pada profil dalam MW tanpa standardisasi dengan k = 2 (Silhouette Score 0,494) membagi hari menjadi *cluster* beban rendah dan beban tinggi yang berkaitan dengan musim dan akhir pekan. Tuning 265 konfigurasi (10 *preprocessing*, 6 algoritma, k = 2 sampai 6) tidak menemukan model yang jelas lebih baik: konfigurasi terpilih hanya sedikit lebih baik daripada konfigurasi awal secara konsisten, dan dipilih sebagai keputusan pertimbangan, bukan keluaran otomatis dari aturan seleksi (yang akan memilih *spectral clustering* dengan selisih +0,007). Karena data membentuk kontinum, *cluster* ini lebih tepat dipakai sebagai ringkasan deskriptif.

Keterbatasan utama adalah tidak adanya data cuaca, pengujian pada satu periode untuk tiap wilayah, dan data yang berasal dari Amerika Serikat. Pengembangan selanjutnya mencakup penambahan prakiraan suhu, prediksi probabilistik, dan validasi pada data beban Indonesia.

## DAFTAR PUSTAKA
[1] Kementerian ESDM, “Paparkan Capaian 2023, Menteri ESDM Klaim Peningkatan Rasio Elektrifikasi,” 15 Jan. 2024. [Online]. Available: https://www.esdm.go.id/id/berita-unit/direktorat-jenderal-ketenagalistrikan/paparkan-capaian-2023-menteri-esdm-klaim-peningkatan-rasio-elektrifikasi (Diakses pada 6 Oktober 2026).

[2] Kementerian ESDM, “Tertinggal dari Negara Lain, Kementerian ESDM Libatkan Stakeholder dalam Diskusi Peningkatan Konsumsi Listrik Per Kapita,” 25 Nov. 2022. [Online]. Available: https://esdm.go.id/id/berita-unit/direktorat-jenderal-ketenagalistrikan/tertinggal-dari-negara-lain-kementerian-esdm-libatkan-stakeholder-dalam-diskusi-peningkatan-konsumsi-listrik-per-kapita (Diakses pada 6 Oktober 2026).

[3] hukumonline, “Bahlil Resmi Umumkan RUPTL Tahun 2025–2034,” 26 Mei 2025. [Online]. Available: https://www.hukumonline.com/berita/a/bahlil-resmi-umumkan-ruptl-tahun-2025-2034-lt68344d6f0dc8a (Diakses pada 8 Oktober 2026).

[4] Investortrust, “PLN Bersiap Hadapi Lonjakan Permintaan Listrik 511 TWh, dari ‘Data Center’ hingga EV,” 2 Jun. 2025. [Online]. Available: https://investortrust.id/business/68024/pln-bersiap-hadapi-lonjakan-permintaan-listrik-511-twh-dari-data-center-hingga-ev (Diakses pada 8 Oktober 2026).

[5] Kementerian ESDM, “Kunjungi APB Ungaran, Menteri ESDM Pastikan Listrik Jawa Bali Aman Jelang Libur Akhir Tahun,” 13 Des. 2018. [Online]. Available: https://esdm.go.id/id/berita-unit/direktorat-jenderal-ketenagalistrikan/kunjungi-apb-ungaran-menteri-esdm-pastikan-listrik-jawa-bali-aman-jelang-libur-akhir-tahun (Diakses pada 11 Oktober 2026).

[6] Bisnis.com, “ESDM Catat Kelebihan Pasokan Listrik Jawa-Bali Turun jadi 4 GW pada 2023,” 18 Jan. 2024. [Online]. Available: https://ekonomi.bisnis.com/read/20240118/44/1733610/esdm-catat-kelebihan-pasokan-listrik-jawa-bali-turun-jadi-4-gw-pada-2023 (Diakses pada 12 Oktober 2026).

[7] T. Hong, “Crystal ball lessons in predictive analytics,” *EnergyBiz*, pp. 35–37, Spring 2015. [Online]. Available: https://blog.drhongtao.com/2015/04/crystal-ball-lessons-in-predictive-analytics.html (Diakses pada 12 Oktober 2026).

[8] M. F. Ainuri, Sarjiya, and I. Ardiyanto, “Short-term load forecasting with long short-term memory: A case study of Java-Bali system,” in *Proc. 12th Int. Conf. Inf. Technol. Electr. Eng. (ICITEE)*, Yogyakarta, Indonesia, 2020, pp. 257–262, doi: 10.1109/ICITEE49829.2020.9271763.

[9] robikscube, “Hourly Energy Consumption,” Kaggle. [Online]. Available: https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption

[10] ashishkumarrana, “PJM Hourly Energy Consumption,” Kaggle. [Online]. Available: https://www.kaggle.com/datasets/ashishkumarrana/pjm-hourly-energy-consumption

[11] “PJM Hourly Energy Consumption,” IEEE DataPort. [Online]. Available: https://ieee-dataport.org/documents/pjm-hourly-energy-consumption

[12] navyasancheti, “PJME-Hourly-Energy-Consumption,” GitHub repository. [Online]. Available: https://github.com/navyasancheti/PJME-Hourly-Energy-Consumption

[13] panambY, “Hourly_Energy_Consumption,” GitHub repository. [Online]. Available: https://github.com/panambY/Hourly_Energy_Consumption

[14] PJM Interconnection, “PJM prepared to meet summer electricity demand,” *Inside Lines*, 11 Mei 2022. [Online]. Available: https://insidelines.pjm.com/?p=205370 (Diakses pada 12 Oktober 2026).

[15] F. Pedregosa et al., “Scikit-learn: Machine learning in Python,” *J. Mach. Learn. Res.*, vol. 12, pp. 2825–2830, 2011.

[16] Python package *holidays*, modul *holidays.US* (kalender libur nasional Amerika Serikat), versi 0.106.

## LAMPIRAN
**Lampiran 1. Dashboard website mini-project AI.** Hasil model dapat dicoba pada dashboard website di https://loads-ai.vercel.app.

**Lampiran 2. Tautan repositori GitHub dan folder Google Drive.**
1. Repositori GitHub: https://github.com/putraparindra/pjmloads-ai-14
2. Folder Google Drive: https://s.id/gOqlo

**Lampiran 3. Grid *hyperparameter* HGB.**

| Parameter | Nilai yang dicoba (PJME, analisis utama) | Nilai yang dicoba (analisis 10 wilayah) | Terbaik pada PJME (analisis utama) |
|---|---|---|---|
| *learning_rate* | 0,03; 0,06 | 0,06 (tetap) | 0,06 |
| *max_iter* | 400; 800 | 400; 800 | 800 |
| *max_leaf_nodes* | 31; 63 | 31; 63 | 31 |
| *min_samples_leaf* | 50; 100 | 50; 100 | 50 |

Pada tabel di atas, grid analisis utama PJME terdiri atas 16 kombinasi, sedangkan grid pada analisis sepuluh wilayah terdiri atas 8 kombinasi karena *learning_rate* dibuat tetap untuk menghemat waktu komputasi. Seluruh *tuning* memakai GridSearchCV dengan TimeSeriesSplit 4 *fold* dan skor MAE pada data *training*.

**Lampiran 4. Hasil *clustering* pada sepuluh wilayah.** Tabel hasil K-Means MW asli pada setiap wilayah dibandingkan dengan konfigurasi awal ditampilkan oleh sel terakhir pada Subbab 3.2.7.

**Lampiran 5. Berkas pendukung.** Seluruh angka pada laporan ini berasal dari keluaran notebook ini. Hasil PJME tersimpan pada `results/results.json` dan hasil sepuluh wilayah pada `results/results_regions.json`. Hasil *tuning clustering* tersimpan pada `cluster_tuning/cluster_tune_PJME.json`, `cluster_pairs_PJME.json`, dan `cluster_tune_<wilayah>.json`, yang dihasilkan oleh skrip `cluster_tune.py`, `cluster_pairs.py`, dan `cluster_regions.py`. Data tiap wilayah berada pada folder `data_pjm/`. *Random seed* yang dipakai adalah 42, dan waktu eksekusi notebook sekitar 40 menit pada 2 CPU (belum termasuk *tuning clustering* penuh yang dijalankan terpisah, sekitar 45 menit per wilayah untuk PJME).

**Lampiran 6. Menyimpan hasil.** Dua sel berikut menyimpan seluruh angka ke `results/results_regions.json` (sepuluh wilayah) dan `results/results.json` (PJME).


In [27]:
json.dump(RES2, open("results/results_regions.json", "w"), indent=1, default=float); print("results/results_regions.json tersimpan")


results/results_regions.json tersimpan


In [28]:
json.dump(RES, open("results/results.json", "w"), indent=1, default=float); print("results/results.json tersimpan")


results/results.json tersimpan
